## Case Study: Bureau Feature and Target Creation 

### Section -1 Feature Engineering
**Read Inquiry, Tradeline and History data and follow the pre-processing steps mentioned below**

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import types as T

In [2]:
spark = (
    SparkSession.builder.appName('tradeline_features')
    .config('spark.driver.memory', '30g')
    .config('spark.sql.legacy.timeParserPolicy', 'LEGACY')
    .config('spark.sql.codegen.wholeStage', 'false')
    .getOrCreate()
)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
spark.conf.set("parquet.enable.dictionary", "false")
spark.conf.set("spark.default.parallelism", 500)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/30 18:15:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/30 18:15:02 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/30 18:15:02 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/30 18:15:02 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/06/30 18:15:02 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/06/30 18:15:02 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
26/06/30 18:15:02 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
26/06/30 18:15:02 WARN Utils: Service 'SparkUI' could not bind on port 4046. Attempting port 4047.


### Input Data Format
#### 1. Feature Engineering Inputs

Load the following parquet files:

```python
trade_var = spark.read.parquet(path + "trade.parquet")
inq_var   = spark.read.parquet(path + "inq.parquet")
ref       = spark.read.parquet(path + "ref_file.parquet")
```

In [6]:
trade_var = spark.read.parquet(path + "oot_trade.parquet")
inq_var   = spark.read.parquet(path + "oot_inq.parquet")
ref       = spark.read.parquet(path + "oot_ref_file.parquet")

In [7]:
trade_var.columns

['ACCOUNT_TYPE_CODE',
 'ACCOUNT_TYPE',
 'REPORTING_MEMBER_NAME',
 'HIGHEST_CREDIT_OR_LOAN_AMOUNT',
 'OPEN_DATE',
 'REPORTING_DATE',
 'CLOSED_DATE',
 'LAST_PAYMENT_DATE',
 'INTEREST_RATE',
 'EMI_AMOUNT',
 'TENURE',
 'CURRENT_BALANCE',
 'AMOUNT_OVERDUE',
 'WRITTEN_OFF_AMOUNT',
 'WRITTEN_OFF_AMOUNT_PRINCIPAL',
 'IS_SUIT_FILED_OR_WILFUL_DEFAULT',
 'IS_WRITTEN_OFF_OR_SETTLED',
 'PAYMENT_HISTORY_START_DATE',
 'PAYMENT_HISTORY_END_DATE',
 'ACCOUNT_HOLDER_TYPE',
 'CREDIT_LIMIT',
 'CASH_LIMIT',
 'ACTUAL_PAYMENT_AMOUNT',
 'COLLATERAL_VALUE',
 'COLLATERAL_TYPE',
 'user_id',
 'payment_history_str']

In [8]:
trade_var.show(3, truncate=False)

26/06/30 18:15:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------------+-------------+---------------------+-----------------------------+----------+--------------+-----------+-----------------+-------------+----------+------+---------------+--------------+------------------+----------------------------+-------------------------------+-------------------------+--------------------------+------------------------+-------------------+------------+----------+---------------------+----------------+---------------+----------+------------------------------------------------------------------------------------------------------------+
|ACCOUNT_TYPE_CODE|ACCOUNT_TYPE |REPORTING_MEMBER_NAME|HIGHEST_CREDIT_OR_LOAN_AMOUNT|OPEN_DATE |REPORTING_DATE|CLOSED_DATE|LAST_PAYMENT_DATE|INTEREST_RATE|EMI_AMOUNT|TENURE|CURRENT_BALANCE|AMOUNT_OVERDUE|WRITTEN_OFF_AMOUNT|WRITTEN_OFF_AMOUNT_PRINCIPAL|IS_SUIT_FILED_OR_WILFUL_DEFAULT|IS_WRITTEN_OFF_OR_SETTLED|PAYMENT_HISTORY_START_DATE|PAYMENT_HISTORY_END_DATE|ACCOUNT_HOLDER_TYPE|CREDIT_LIMIT|CASH_LIMIT|ACTUAL_PAYME

In [9]:
trade_var.select("ACCOUNT_HOLDER_TYPE").show(3)

+-------------------+
|ACCOUNT_HOLDER_TYPE|
+-------------------+
|         individual|
|         individual|
|         individual|
+-------------------+
only showing top 3 rows



In [10]:
# Just view the length for each row (payment_history_str)
df=trade_var.select(
    F.length("payment_history_str"),
    (F.length("payment_history_str") / 3)
)
df.show(10)

+---------------------------+---------------------------------+
|length(payment_history_str)|(length(payment_history_str) / 3)|
+---------------------------+---------------------------------+
|                        108|                             36.0|
|                        108|                             36.0|
|                        108|                             36.0|
|                        108|                             36.0|
|                        108|                             36.0|
|                        108|                             36.0|
|                        108|                             36.0|
|                        108|                             36.0|
|                        108|                             36.0|
|                        108|                             36.0|
+---------------------------+---------------------------------+
only showing top 10 rows



In [11]:
df.select("length(payment_history_str)").distinct().count()

2

In [12]:
df.groupBy("length(payment_history_str)").count().show(truncate=False)

+---------------------------+-------+
|length(payment_history_str)|count  |
+---------------------------+-------+
|108                        |1174727|
|NULL                       |7788   |
+---------------------------+-------+



In [13]:
trade_var.count()

1182515

In [14]:
inq_var.show(5)

+---------------+-----------------+-------------+--------------+-----------+
|DATE_OF_ENQUIRY|ACCOUNT_TYPE_CODE| ACCOUNT_TYPE|ENQUIRY_AMOUNT|    user_id|
+---------------+-----------------+-------------+--------------+-----------+
|     2024-06-11|               06|Consumer Loan|       10000.0|17179871003|
|     2022-10-09|               05|Personal Loan|       25000.0|17179871003|
|     2025-02-20|               06|Consumer Loan|       55000.0|17179871003|
|     2023-08-18|               10|  Credit Card|       20000.0|17179871003|
|     2024-10-28|               10|  Credit Card|        1000.0|17179871003|
+---------------+-----------------+-------------+--------------+-----------+
only showing top 5 rows



In [15]:
ref.show(7)

+-----------+----------+
|    user_id|retro_date|
+-----------+----------+
|42949684744|2025-03-08|
|17179872727|2025-03-09|
|17179870731|2025-03-09|
|       2067|2025-04-05|
|42949674682|2025-03-10|
|25769812768|2025-04-02|
|17179873286|2025-03-11|
+-----------+----------+
only showing top 7 rows



In [16]:
ref.select("retro_date").distinct().count()

90

In [17]:
trade_var.printSchema()

root
 |-- ACCOUNT_TYPE_CODE: string (nullable = true)
 |-- ACCOUNT_TYPE: string (nullable = true)
 |-- REPORTING_MEMBER_NAME: string (nullable = true)
 |-- HIGHEST_CREDIT_OR_LOAN_AMOUNT: double (nullable = true)
 |-- OPEN_DATE: date (nullable = true)
 |-- REPORTING_DATE: date (nullable = true)
 |-- CLOSED_DATE: date (nullable = true)
 |-- LAST_PAYMENT_DATE: date (nullable = true)
 |-- INTEREST_RATE: double (nullable = true)
 |-- EMI_AMOUNT: double (nullable = true)
 |-- TENURE: string (nullable = true)
 |-- CURRENT_BALANCE: double (nullable = true)
 |-- AMOUNT_OVERDUE: double (nullable = true)
 |-- WRITTEN_OFF_AMOUNT: double (nullable = true)
 |-- WRITTEN_OFF_AMOUNT_PRINCIPAL: double (nullable = true)
 |-- IS_SUIT_FILED_OR_WILFUL_DEFAULT: decimal(38,0) (nullable = true)
 |-- IS_WRITTEN_OFF_OR_SETTLED: decimal(38,0) (nullable = true)
 |-- PAYMENT_HISTORY_START_DATE: date (nullable = true)
 |-- PAYMENT_HISTORY_END_DATE: date (nullable = true)
 |-- ACCOUNT_HOLDER_TYPE: string (nullable = 

In [18]:
inq_var.printSchema()

root
 |-- DATE_OF_ENQUIRY: date (nullable = true)
 |-- ACCOUNT_TYPE_CODE: string (nullable = true)
 |-- ACCOUNT_TYPE: string (nullable = true)
 |-- ENQUIRY_AMOUNT: double (nullable = true)
 |-- user_id: long (nullable = true)



In [19]:
ref.printSchema()

root
 |-- user_id: long (nullable = true)
 |-- retro_date: date (nullable = true)



### Tradeline

* Each row represents one trade/account for a customer.
* Multiple rows can exist for the same `user_id`.
* `payment_history_str` contains historical repayment behaviour.
* Date columns should be converted to proper date format before fe

### Enquiry

* Each row represents one enquiry event.
* Multiple enquiries can exist for the same customer.
* Enquiry features are generally created using lookback windows from `retro_date`


### Reference

* Features should be created relative to `retro_date`.
* `retro_date` acts as the observation/reference date.
* Only information available on or before retro_date should be used

## Target Creation
### Data for Target creation:

```python
df_tar = spark.read.parquet(out_path + "target_file.parquet")
```

In [20]:
#df_tar = spark.read.parquet(out_path + "target_file.parquet")

#### `Target Creation Steps`

1. Read base_data and join on Raw Tradeline data on common reference_no in both
2. Format all date columns and set dates ≤ 1950-01-01 to null 
3. Filters:
   - Date Reported and Date Opened cannot be null
   - Date Opened <= Date Reported
   - Date Opened  <= Date Closed or Date Closed should be null
4. DPD Bucket creation (refer to doc) and save the intermediate TL file 
5. Target Creation
    - Create intermediate columns as in doc attached to get `user_ever60_9m` 
    - Filters:
        - records where trades are live and clean at retro_date
        - records where dpd from retro 6m is not null
        - only PL, BL products #with sanction amount>= 15K
6. For each horizon (6, 9, or 12 months after retro_date), we look at how many of those months actually have a non-null DPD value reported.If it is 3 or more, we compute the flag normally, else if less than 3, we don't have enough information to call it either way, so the flag is set to NULL, not 0.
7. Keep only which are not null

### Additional Pre-processing Instructions

#### 1. Date and Numeric Formatting

* Convert all date columns into proper date format.
* Convert numeric columns into appropriate numeric types.
* Any date earlier than `1950-01-01` should be replaced with `NULL`.


### 2. Account Type Mapping

* Create standardized `ACCOUNT_TYPE` using the mapping provided in the Excel sheet.
* Mapping should be done using `ACCOUNT_TYPE_CODE`.
* Ensure all unknown account types are tagged appropriately.


#### 3. Inquiry Deduplication

Deduplicate inquiry records using the following logic:
iry records using the following logic:

* Duplicate definition:

  * Same `user_id`
  * Same `DATE_OF_ENQUIRY`
  * Same `ACCOUNT_TYPE`

* Keep only the record having the maximum `ENQUIRY_AMOUNT`.

#### 4. Mandatory Inquiry Filters

Apply the following filters before feature creation:

**Required Conditions**

* `retro_date` cannot be null
* `DATE_OF_ENQUIRY` cannot be null
* `retro_date > DATE_OF_ENQUIRY`

**Lookback Window**

* Consider only inquiries falling within the required historical lookback period from `retro_date`.
* Example windows:

  * Last 1 Month
  * Last 3 Months
  * Last 6 Months
  * Last 12 Months


#### 5. Trade Filters

Apply the following validations on trade data:

* `retro_date` cannot be null
* `REPORTING_DATE` cannot be null
* Use only trades where:

```text
REPORTING_DATE <= retro_date
```

* Exclude invalid or corrupted trade records wherever necessary.


#### 6. Payment History Handling

* `payment_history_str` contains repayment behaviour history.
* Parse the string carefully before deriving DPD or delinquency features.
* Handle missing or malformed payment history values safely.


## Important Guidelines



### Leakage Prevention

While creating features:

* Use only records with dates <= `retro_date`
* Do not use future information
* Ensure all aggregations are backward-looking


## Expected Workflow

### Step 1

Load parquet files.

### Step 2

Convert date columns.

### Step 3

Filter trades/enquiries using `retro_date`.

### Step 4

Create customer-level aggregated features.

### Step 5

Join features with target/reference population.

### Step 6

Prepare final model dataset.



# Final Output

The final feature table should contain:

* One row per `user_id`
* Features created using historical bureau information
* No future leakage beyond `retro_date`
* Target


In [21]:
trade = trade_var.join(ref, on="user_id", how="left")

In [22]:
trade.printSchema()

root
 |-- user_id: long (nullable = true)
 |-- ACCOUNT_TYPE_CODE: string (nullable = true)
 |-- ACCOUNT_TYPE: string (nullable = true)
 |-- REPORTING_MEMBER_NAME: string (nullable = true)
 |-- HIGHEST_CREDIT_OR_LOAN_AMOUNT: double (nullable = true)
 |-- OPEN_DATE: date (nullable = true)
 |-- REPORTING_DATE: date (nullable = true)
 |-- CLOSED_DATE: date (nullable = true)
 |-- LAST_PAYMENT_DATE: date (nullable = true)
 |-- INTEREST_RATE: double (nullable = true)
 |-- EMI_AMOUNT: double (nullable = true)
 |-- TENURE: string (nullable = true)
 |-- CURRENT_BALANCE: double (nullable = true)
 |-- AMOUNT_OVERDUE: double (nullable = true)
 |-- WRITTEN_OFF_AMOUNT: double (nullable = true)
 |-- WRITTEN_OFF_AMOUNT_PRINCIPAL: double (nullable = true)
 |-- IS_SUIT_FILED_OR_WILFUL_DEFAULT: decimal(38,0) (nullable = true)
 |-- IS_WRITTEN_OFF_OR_SETTLED: decimal(38,0) (nullable = true)
 |-- PAYMENT_HISTORY_START_DATE: date (nullable = true)
 |-- PAYMENT_HISTORY_END_DATE: date (nullable = true)
 |-- ACC

In [23]:
#Format all date columns and set dates ≤ 1950-01-01 to null

In [24]:
date_cols = [
    "OPEN_DATE",
    "REPORTING_DATE",
    "CLOSED_DATE",
    "LAST_PAYMENT_DATE",
    "PAYMENT_HISTORY_START_DATE",
    "PAYMENT_HISTORY_END_DATE",
    "retro_date"
]

In [25]:
for col_name in date_cols:
    trade = trade.withColumn(
        col_name,
        F.when(
            F.col(col_name) <= F.lit("1950-01-01").cast("date"),
            F.lit(None).cast("date")
        ).otherwise(F.col(col_name))
    )

In [26]:
trade.show(1)

+----------+-----------------+-------------+---------------------+-----------------------------+----------+--------------+-----------+-----------------+-------------+----------+------+---------------+--------------+------------------+----------------------------+-------------------------------+-------------------------+--------------------------+------------------------+-------------------+------------+----------+---------------------+----------------+---------------+--------------------+----------+
|   user_id|ACCOUNT_TYPE_CODE| ACCOUNT_TYPE|REPORTING_MEMBER_NAME|HIGHEST_CREDIT_OR_LOAN_AMOUNT| OPEN_DATE|REPORTING_DATE|CLOSED_DATE|LAST_PAYMENT_DATE|INTEREST_RATE|EMI_AMOUNT|TENURE|CURRENT_BALANCE|AMOUNT_OVERDUE|WRITTEN_OFF_AMOUNT|WRITTEN_OFF_AMOUNT_PRINCIPAL|IS_SUIT_FILED_OR_WILFUL_DEFAULT|IS_WRITTEN_OFF_OR_SETTLED|PAYMENT_HISTORY_START_DATE|PAYMENT_HISTORY_END_DATE|ACCOUNT_HOLDER_TYPE|CREDIT_LIMIT|CASH_LIMIT|ACTUAL_PAYMENT_AMOUNT|COLLATERAL_VALUE|COLLATERAL_TYPE| payment_history_str|re

In [27]:
# Filters:
# Date Reported and Date Opened cannot be null
# Date Opened <= Date Reported
# Date Opened <= Date Closed or Date Closed should be null

In [28]:
# Trade Filters
# Apply the following validations on trade data:

# retro_date cannot be null
# REPORTING_DATE cannot be null
# Use only trades where:
# REPORTING_DATE <= retro_date
# Exclude invalid or corrupted trade records wherever necessary.

In [29]:
trade = trade.filter(
    F.col("REPORTING_DATE").isNotNull() &
    F.col("OPEN_DATE").isNotNull() &
    (F.col("OPEN_DATE") <= F.col("REPORTING_DATE")) &
    ((F.col("CLOSED_DATE").isNull()) | (F.col("OPEN_DATE") <= F.col("CLOSED_DATE")))
)

In [30]:
# Filter trades: only those reported before or on retro_date
trade = trade.filter(F.col("REPORTING_DATE") <= F.col("retro_date"))

In [31]:
# Filters:
# retro_date > OPEN_DATE
# retro_date > PAYMENT_HISTORY_END_DATE & retro_date > PAYMENT_HISTORY_START_DATE
# retro_date, OPEN_DATE, PAYMENT_HISTORY_START_DATE NOT NULL
# CLOSED_DATE > OPEN_DATE OR CLOSED_DATE IS NULL
# Remove hanging trades: (retro_date - REPORTING_DATE) <= 36 months

In [32]:
trade = trade.filter(
    F.col("retro_date").isNotNull() &
    F.col("OPEN_DATE").isNotNull() &
    F.col("PAYMENT_HISTORY_START_DATE").isNotNull() &
    (F.col("retro_date") > F.col("OPEN_DATE")) &
    (F.col("retro_date") > F.col("PAYMENT_HISTORY_START_DATE")) &
    (F.col("retro_date") > F.col("PAYMENT_HISTORY_END_DATE")) &
    (
        (F.col("CLOSED_DATE") > F.col("OPEN_DATE")) |
        F.col("CLOSED_DATE").isNull()
    ) &
    (
        F.col("REPORTING_DATE") >=
        F.add_months(F.col("retro_date"), -36)
    )
)

In [33]:
# CURRENT_BALANCE, SANCTIONED_AMOUNT, HIGHEST_CREDIT_OR_LOAN_AMOUNT, AMOUNT_OVERDUE: negatives -> 0
# CREDIT_LIMIT: negatives -> NULL

In [34]:
trade = trade.withColumn(
    "CURRENT_BALANCE",
    F.when(F.col("CURRENT_BALANCE") < 0, F.lit(0)).otherwise(F.col("CURRENT_BALANCE"))
).withColumn(
    "HIGHEST_CREDIT_OR_LOAN_AMOUNT",
    F.when(F.col("HIGHEST_CREDIT_OR_LOAN_AMOUNT") < 0, F.lit(0)).otherwise(F.col("HIGHEST_CREDIT_OR_LOAN_AMOUNT"))
).withColumn(
    "AMOUNT_OVERDUE",
    F.when(F.col("AMOUNT_OVERDUE") < 0, F.lit(0)).otherwise(F.col("AMOUNT_OVERDUE"))
).withColumn(
    "CREDIT_LIMIT",
    F.when(F.col("CREDIT_LIMIT") < 0, F.lit(None)).otherwise(F.col("CREDIT_LIMIT"))
)

In [35]:
trade.count()

979919

In [36]:
enquiries = inq_var.join(ref, on="user_id", how="left")

In [37]:
enquiries.show(1)

+-----------+---------------+-----------------+-------------+--------------+----------+
|    user_id|DATE_OF_ENQUIRY|ACCOUNT_TYPE_CODE| ACCOUNT_TYPE|ENQUIRY_AMOUNT|retro_date|
+-----------+---------------+-----------------+-------------+--------------+----------+
|17179871003|     2024-06-11|               06|Consumer Loan|       10000.0|2025-04-05|
+-----------+---------------+-----------------+-------------+--------------+----------+
only showing top 1 row



In [38]:
# Required Conditions:

# retro_date cannot be null
# DATE_OF_ENQUIRY cannot be null
# retro_date > DATE_OF_ENQUIRY
# Any date earlier than 1950-01-01 should be replaced with NULL

In [39]:
enquiries = enquiries.withColumn(
    "DATE_OF_ENQUIRY",
    F.when(
        F.col("DATE_OF_ENQUIRY") <= F.lit("1950-01-01").cast("date"),
        F.lit(None).cast("date")
    ).otherwise(F.col("DATE_OF_ENQUIRY"))
)

enquiries = enquiries.withColumn(
    "retro_date",
    F.when(
        F.col("retro_date") <= F.lit("1950-01-01").cast("date"),
        F.lit(None).cast("date")
    ).otherwise(F.col("retro_date"))
)

In [40]:
enquiries = enquiries.filter(
    F.col("retro_date").isNotNull() &
    F.col("DATE_OF_ENQUIRY").isNotNull() &
    (F.col("retro_date") > F.col("DATE_OF_ENQUIRY"))
)

In [41]:
enquiries.count()

2085565

In [42]:
# 3. Inquiry Deduplication
# Deduplicate inquiry records using the following logic: iry records using the following logic:

# Duplicate definition:

# Same user_id
# Same DATE_OF_ENQUIRY
# Same ACCOUNT_TYPE
# Keep only the record having the maximum ENQUIRY_AMOUNT.

In [43]:
# Dedepulication: Drop all duplicate records of Inquiry in a Day of same account type keeping the one with Maximum Enquiry Amount

In [44]:
w = Window.partitionBy(
    "user_id", 
    "DATE_OF_ENQUIRY", 
    "ACCOUNT_TYPE"
).orderBy(F.col("ENQUIRY_AMOUNT").desc())

enquiries = enquiries.withColumn(
    "row_num", 
    F.row_number().over(w)
).filter(F.col("row_num") == 1).drop("row_num")

In [45]:
enquiries.count()

1858754

In [46]:
enquiries.show()

+-------+---------------+-----------------+--------------------+--------------+----------+
|user_id|DATE_OF_ENQUIRY|ACCOUNT_TYPE_CODE|        ACCOUNT_TYPE|ENQUIRY_AMOUNT|retro_date|
+-------+---------------+-----------------+--------------------+--------------+----------+
|      0|     2023-09-20|               05|       Personal Loan|        5000.0|2025-03-29|
|      0|     2023-09-23|               05|       Personal Loan|        1500.0|2025-03-29|
|      3|     2024-09-29|               05|       Personal Loan|        5000.0|2025-04-07|
|      4|     2024-07-27|               05|       Personal Loan|        3000.0|2025-04-11|
|      7|     2024-10-01|               05|       Personal Loan|      100000.0|2025-04-05|
|      7|     2025-02-14|               34|        Tractor Loan|      322719.0|2025-04-05|
|      8|     2022-08-01|               10|         Credit Card|         100.0|2025-03-09|
|      8|     2022-08-29|               05|       Personal Loan|        4000.0|2025-03-09|

In [47]:
# 2. Account Type Mapping
# Create standardized ACCOUNT_TYPE using the mapping provided in the Excel sheet.
# Mapping should be done using ACCOUNT_TYPE_CODE.
# Ensure all unknown account types are tagged appropriately.

In [48]:
trade.show(1)

+----------+-----------------+-------------+---------------------+-----------------------------+----------+--------------+-----------+-----------------+-------------+----------+------+---------------+--------------+------------------+----------------------------+-------------------------------+-------------------------+--------------------------+------------------------+-------------------+------------+----------+---------------------+----------------+---------------+--------------------+----------+
|   user_id|ACCOUNT_TYPE_CODE| ACCOUNT_TYPE|REPORTING_MEMBER_NAME|HIGHEST_CREDIT_OR_LOAN_AMOUNT| OPEN_DATE|REPORTING_DATE|CLOSED_DATE|LAST_PAYMENT_DATE|INTEREST_RATE|EMI_AMOUNT|TENURE|CURRENT_BALANCE|AMOUNT_OVERDUE|WRITTEN_OFF_AMOUNT|WRITTEN_OFF_AMOUNT_PRINCIPAL|IS_SUIT_FILED_OR_WILFUL_DEFAULT|IS_WRITTEN_OFF_OR_SETTLED|PAYMENT_HISTORY_START_DATE|PAYMENT_HISTORY_END_DATE|ACCOUNT_HOLDER_TYPE|CREDIT_LIMIT|CASH_LIMIT|ACTUAL_PAYMENT_AMOUNT|COLLATERAL_VALUE|COLLATERAL_TYPE| payment_history_str|re

In [49]:
mapping_df = spark.read.csv("Feature_Account_Type_Final.csv", header=True, inferSchema=True)

In [50]:
mapping_df.show(2)

+-----+-------+--------------------+--------------+---------+-------+
|VALUE|    USE|        ACCOUNT_TYPE|BROAD_CATEGORY|REVOLVING|SECURED|
+-----+-------+--------------------+--------------+---------+-------+
|    0|Current|               Other|         Other|        0|      0|
|    1|Current|Auto Loan (Personal)|            AL|        0|      1|
+-----+-------+--------------------+--------------+---------+-------+
only showing top 2 rows



In [51]:
# Create standardized ACCOUNT_TYPE using the mapping provided in the Excel sheet.
# Mapping should be done using ACCOUNT_TYPE_CODE.

In [52]:
# Rename columns and ensure correct types
mapping_df = mapping_df.select(
    F.col("VALUE").alias("ACCOUNT_TYPE_CODE"),
    F.col("BROAD_CATEGORY"),
    F.col("REVOLVING").cast("int"),
    F.col("SECURED").cast("int")
)

In [53]:
mapping_bc = F.broadcast(mapping_df)

In [54]:
trade = trade.join(mapping_bc, on="ACCOUNT_TYPE_CODE", how="left")

In [55]:
# Ensure all unknown account types are tagged appropriately.

In [56]:
trade = trade.withColumn("BROAD_CATEGORY", F.coalesce(F.col("BROAD_CATEGORY"), F.lit("Other")))
trade = trade.withColumn("REVOLVING", F.coalesce(F.col("REVOLVING"), F.lit(0)))
trade = trade.withColumn("SECURED", F.coalesce(F.col("SECURED"), F.lit(0)))

In [57]:
len(trade.columns)

31

In [58]:
enquiries = enquiries.join(mapping_bc, on="ACCOUNT_TYPE_CODE", how="left")

In [59]:
enquiries.printSchema()

root
 |-- ACCOUNT_TYPE_CODE: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- DATE_OF_ENQUIRY: date (nullable = true)
 |-- ACCOUNT_TYPE: string (nullable = true)
 |-- ENQUIRY_AMOUNT: double (nullable = true)
 |-- retro_date: date (nullable = true)
 |-- BROAD_CATEGORY: string (nullable = true)
 |-- REVOLVING: integer (nullable = true)
 |-- SECURED: integer (nullable = true)



In [60]:
#Inquiries with ACCOUNT_TYPE_CODE >= 62 are soft inquiries and are removed

In [61]:
enquiries = enquiries.filter(
    F.col("ACCOUNT_TYPE_CODE").isNotNull() &
    (F.col("ACCOUNT_TYPE_CODE").cast("int") < 62)
)

In [62]:
enquiries = enquiries.filter(
    F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)
)

In [63]:
enquiries = enquiries.withColumn("BROAD_CATEGORY", F.coalesce(F.col("BROAD_CATEGORY"), F.lit("Other")))
enquiries = enquiries.withColumn("REVOLVING", F.coalesce(F.col("REVOLVING"), F.lit(0)))
enquiries = enquiries.withColumn("SECURED", F.coalesce(F.col("SECURED"), F.lit(0)))

In [64]:
enquiries.printSchema()

root
 |-- ACCOUNT_TYPE_CODE: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- DATE_OF_ENQUIRY: date (nullable = true)
 |-- ACCOUNT_TYPE: string (nullable = true)
 |-- ENQUIRY_AMOUNT: double (nullable = true)
 |-- retro_date: date (nullable = true)
 |-- BROAD_CATEGORY: string (nullable = false)
 |-- REVOLVING: integer (nullable = false)
 |-- SECURED: integer (nullable = false)



In [65]:
enquiries.show()

+-----------------+-------+---------------+--------------------+--------------+----------+--------------+---------+-------+
|ACCOUNT_TYPE_CODE|user_id|DATE_OF_ENQUIRY|        ACCOUNT_TYPE|ENQUIRY_AMOUNT|retro_date|BROAD_CATEGORY|REVOLVING|SECURED|
+-----------------+-------+---------------+--------------------+--------------+----------+--------------+---------+-------+
|               05|      0|     2023-09-20|       Personal Loan|        5000.0|2025-03-29|            PL|        0|      0|
|               05|      0|     2023-09-23|       Personal Loan|        1500.0|2025-03-29|            PL|        0|      0|
|               05|      3|     2024-09-29|       Personal Loan|        5000.0|2025-04-07|            PL|        0|      0|
|               05|      4|     2024-07-27|       Personal Loan|        3000.0|2025-04-11|            PL|        0|      0|
|               05|      7|     2024-10-01|       Personal Loan|      100000.0|2025-04-05|            PL|        0|      0|
|       

## Lookback Window

In [66]:
# inquiry_features = enquiries.groupBy("user_id").agg(
#     F.sum(F.when(F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -1), 1).otherwise(0)).alias("TOTAL_INQ_1M"),
#     F.sum(F.when(F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3), 1).otherwise(0)).alias("TOTAL_INQ_3M"),
#     F.sum(F.when(F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -6), 1).otherwise(0)).alias("TOTAL_INQ_6M"),
#     F.sum(F.when(F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12), 1).otherwise(0)).alias("TOTAL_INQ_12M"),
# )

In [67]:
# inquiry_features.show()

## payment_history_str and DPD String

In [68]:
trade_var.select(
    "payment_history_str",
    F.substring("payment_history_str", 1, 3).alias("first_3_digits")
).show(20, truncate=False)

+------------------------------------------------------------------------------------------------------------+--------------+
|payment_history_str                                                                                         |first_3_digits|
+------------------------------------------------------------------------------------------------------------+--------------+
|000001000000001001000000XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX|000           |
|000000000000000000000000000000XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX|000           |
|000000000000000000000000000000000000000000000000000000000000000000000000000000000000XXXXXXXXXXXXXXXXXXXXXXXX|000           |
|000000000000000000000000000XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX|000           |
|000000000000XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX|000     

In [69]:
trade.select(F.col("PAYMENT_HISTORY_START_DATE"), F.col("PAYMENT_HISTORY_END_DATE"), F.col("retro_date")).show(3)

+--------------------------+------------------------+----------+
|PAYMENT_HISTORY_START_DATE|PAYMENT_HISTORY_END_DATE|retro_date|
+--------------------------+------------------------+----------+
|                2022-07-01|              2021-11-01|2025-03-08|
|                2022-08-01|              2021-10-01|2025-03-08|
|                2024-01-01|              2021-09-01|2025-03-08|
+--------------------------+------------------------+----------+
only showing top 3 rows



In [70]:
trade_var.select(
    F.substring("payment_history_str", 1, 3).alias("first_3_digits")
).distinct().count()

896

In [71]:
trade_var.select(
    F.substring("payment_history_str", 1, 3).alias("first_3_digits")
).distinct().orderBy("first_3_digits").show(truncate=False)

+--------------+
|first_3_digits|
+--------------+
|NULL          |
|000           |
|001           |
|002           |
|003           |
|004           |
|005           |
|006           |
|007           |
|008           |
|009           |
|010           |
|011           |
|012           |
|013           |
|014           |
|015           |
|016           |
|017           |
|018           |
+--------------+
only showing top 20 rows



In [72]:
trade_var.filter(
    F.substring("payment_history_str", 1, 3) == "XXX"
).count()

0

In [73]:
trade_var.select(
    F.expr("substring(payment_history_str, length(payment_history_str)-2, 3)")
     .alias("last_3_digits")
).distinct().show(truncate=False)

+-------------+
|last_3_digits|
+-------------+
|NULL         |
|XXX          |
+-------------+



In [74]:
trade.select(
    "PAYMENT_HISTORY_START_DATE",
    "PAYMENT_HISTORY_END_DATE",
    F.months_between(F.col("PAYMENT_HISTORY_END_DATE"), F.col("PAYMENT_HISTORY_START_DATE")).alias("months_diff"),
    "payment_history_str" 
).show(20, truncate=False)

+--------------------------+------------------------+-----------+------------------------------------------------------------------------------------------------------------+
|PAYMENT_HISTORY_START_DATE|PAYMENT_HISTORY_END_DATE|months_diff|payment_history_str                                                                                         |
+--------------------------+------------------------+-----------+------------------------------------------------------------------------------------------------------------+
|2022-07-01                |2021-11-01              |-8.0       |000001000000001001000000XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX|
|2022-08-01                |2021-10-01              |-10.0      |000000000000000000000000000000XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX|
|2024-01-01                |2021-09-01              |-28.0      |000000000000000000000000000000000000000000000000000000000000

In [75]:
trade.filter(
    F.months_between(F.col("PAYMENT_HISTORY_END_DATE"), F.col("PAYMENT_HISTORY_START_DATE")) <= -36
).select(
    "PAYMENT_HISTORY_START_DATE",
    "PAYMENT_HISTORY_END_DATE",
    F.months_between(F.col("PAYMENT_HISTORY_END_DATE"), F.col("PAYMENT_HISTORY_START_DATE")).alias("months_diff"),
    "payment_history_str"
).show(20, truncate=False)

+--------------------------+------------------------+-----------+------------------------------------------------------------------------------------------------------------+
|PAYMENT_HISTORY_START_DATE|PAYMENT_HISTORY_END_DATE|months_diff|payment_history_str                                                                                         |
+--------------------------+------------------------+-----------+------------------------------------------------------------------------------------------------------------+
|2024-03-01                |2021-03-01              |-36.0      |214214214214214214214214214214214214214214214214214000000000900900900900900900900900900900900000900900900XXX|
|2024-03-01                |2021-03-01              |-36.0      |000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000XXX|
|2025-01-01                |2022-01-01              |-36.0      |000000000000000000000000000000000000000000000000000000000000

In [76]:
# Count rows where END < START
trade.filter(
    F.col("PAYMENT_HISTORY_END_DATE") < F.col("PAYMENT_HISTORY_START_DATE")
).count()

872501

In [77]:
trade.count()

979919

In [78]:
trade.agg(
    F.sum(F.when(F.col("retro_date") < F.col("PAYMENT_HISTORY_START_DATE"), 1).otherwise(0)).alias("retro_date < PAYMENT_HISTORY_START_DATE"),
    F.sum(F.when(F.col("retro_date") > F.col("PAYMENT_HISTORY_START_DATE"), 1).otherwise(0)).alias("retro_date > PAYMENT_HISTORY_START_DATE"),
    F.sum(F.when(F.col("retro_date") == F.col("PAYMENT_HISTORY_START_DATE"), 1).otherwise(0)).alias("retro_date = PAYMENT_HISTORY_START_DATE"),
    F.sum(F.when(F.col("retro_date").isNull() | F.col("PAYMENT_HISTORY_START_DATE").isNull(), 1).otherwise(0)).alias("NULL_DATES")
).show(truncate=False)

+---------------------------------------+---------------------------------------+---------------------------------------+----------+
|retro_date < PAYMENT_HISTORY_START_DATE|retro_date > PAYMENT_HISTORY_START_DATE|retro_date = PAYMENT_HISTORY_START_DATE|NULL_DATES|
+---------------------------------------+---------------------------------------+---------------------------------------+----------+
|0                                      |979919                                 |0                                      |0         |
+---------------------------------------+---------------------------------------+---------------------------------------+----------+



In [79]:
#       2016-04-01  .................... 2024-09-01  ................... 2025-03-23
# (PAYMENT_HISTORY_END_DATE)     (PAYMENT_HISTORY_START_DATE)           (retro_date)
# (OLDEST metadata)               (NEWEST string char)                   (Snapshot)  

In [80]:
trade.select(
    "PAYMENT_HISTORY_END_DATE",
    "PAYMENT_HISTORY_START_DATE",
    "retro_date"
).show(5, truncate=False)

+------------------------+--------------------------+----------+
|PAYMENT_HISTORY_END_DATE|PAYMENT_HISTORY_START_DATE|retro_date|
+------------------------+--------------------------+----------+
|2021-11-01              |2022-07-01                |2025-03-08|
|2021-10-01              |2022-08-01                |2025-03-08|
|2021-09-01              |2024-01-01                |2025-03-08|
|2024-06-01              |2024-10-01                |2025-03-08|
|2023-11-01              |2025-02-01                |2025-03-08|
+------------------------+--------------------------+----------+
only showing top 5 rows



In [81]:
trade.select(
    "PAYMENT_HISTORY_START_DATE",
    "retro_date",
    F.months_between(F.col("retro_date"), F.col("PAYMENT_HISTORY_START_DATE")).alias("months_diff"),
    "payment_history_str"
).show(5, truncate=False)

+--------------------------+----------+-----------+------------------------------------------------------------------------------------------------------------+
|PAYMENT_HISTORY_START_DATE|retro_date|months_diff|payment_history_str                                                                                         |
+--------------------------+----------+-----------+------------------------------------------------------------------------------------------------------------+
|2022-07-01                |2025-03-08|32.22580645|000001000000001001000000XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX|
|2022-08-01                |2025-03-08|31.22580645|000000000000000000000000000000XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX|
|2024-01-01                |2025-03-08|14.22580645|000000000000000000000000000000000000000000000000000000000000000000000000000000000000XXXXXXXXXXXXXXXXXXXXXXXX|
|2024-10-01                |2025-0

In [82]:
trade.select(
    "PAYMENT_HISTORY_START_DATE",
    "retro_date",
    F.months_between(F.col("retro_date"), F.col("PAYMENT_HISTORY_START_DATE")).alias("months_diff"),
).show(5, truncate=False)

+--------------------------+----------+-----------+
|PAYMENT_HISTORY_START_DATE|retro_date|months_diff|
+--------------------------+----------+-----------+
|2022-07-01                |2025-03-08|32.22580645|
|2022-08-01                |2025-03-08|31.22580645|
|2024-01-01                |2025-03-08|14.22580645|
|2024-10-01                |2025-03-08|5.22580645 |
|2025-02-01                |2025-03-08|1.22580645 |
+--------------------------+----------+-----------+
only showing top 5 rows



In [83]:
trade.select(
    "OPEN_DATE",
    "CLOSED_DATE",
    "PAYMENT_HISTORY_END_DATE",
    "PAYMENT_HISTORY_START_DATE",
    "REPORTING_DATE",
    "payment_history_str"
).show(5, truncate=False)

+----------+-----------+------------------------+--------------------------+--------------+------------------------------------------------------------------------------------------------------------+
|OPEN_DATE |CLOSED_DATE|PAYMENT_HISTORY_END_DATE|PAYMENT_HISTORY_START_DATE|REPORTING_DATE|payment_history_str                                                                                         |
+----------+-----------+------------------------+--------------------------+--------------+------------------------------------------------------------------------------------------------------------+
|2021-11-26|2022-07-29 |2021-11-01              |2022-07-01                |2022-07-31    |000001000000001001000000XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX|
|2021-10-23|2022-08-17 |2021-10-01              |2022-08-01                |2022-08-31    |000000000000000000000000000000XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX

In [84]:
trade.agg(
    F.sum(F.when(F.col("PAYMENT_HISTORY_START_DATE").isNull(), 1).otherwise(0)).alias("null_start_date"),
    F.sum(F.when(F.col("retro_date").isNull(), 1).otherwise(0)).alias("null_retro_date")
).show()

+---------------+---------------+
|null_start_date|null_retro_date|
+---------------+---------------+
|              0|              0|
+---------------+---------------+



In [85]:
# Step 1: split original paymenthistorystr into 3-character codes
trade = trade.withColumn(
    "raw_codes",
    F.when(
        F.col("payment_history_str").isNull(),
        F.lit(None)
    ).otherwise(
        F.expr("regexp_extract_all(payment_history_str, '.{3}', 0)")
    )
)

In [86]:
trade.select("payment_history_str", "raw_codes").show(1,truncate=False)

+------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|payment_history_str                                                                                         |raw_codes                                                                                                                                                                           |
+------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|000001000000001001000000XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX|[000, 001, 000

In [87]:
# Step 2: Reverse the array of 3-character codes
trade = trade.withColumn(
    "raw_codes",
    # F.reverse(F.col("raw_codes"))
    F.col("raw_codes")
)

In [88]:
trade.select("raw_codes").show(2,truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|raw_codes                                                                                                                                                                           |
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[000, 001, 000, 000, 001, 001, 000, 000, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX]|
|[000, 000, 000, 000, 000, 000, 000, 000, 000, 000, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX, XXX]|
+------------------------------------------------------------------------------------

In [89]:
# Count rows where raw_codes is null 

In [90]:
trade.filter(F.col("raw_codes").isNull()).count()

6523

In [91]:
# Count rows where payment_history_str is null 

In [92]:
trade.filter(F.col("payment_history_str").isNull()).count()

6523

In [93]:
# Count rows where raw_codes length != 36
trade.filter(F.size("raw_codes") != 36).count()

6523

In [94]:
trade.groupBy(
    F.size("raw_codes").alias("array_length")
).count().orderBy("array_length").show(truncate=False)

+------------+------+
|array_length|count |
+------------+------+
|-1          |6523  |
|36          |973396|
+------------+------+



In [95]:
# So, for non-null, number of codes is 36

In [96]:
# Step 3: Compute the month difference ---
trade = trade.withColumn(
    "diff",
    F.months_between(F.col("retro_date"), F.col("PAYMENT_HISTORY_START_DATE")).cast("int")
)

In [97]:
# --- Step 4: Construct the new 36-code array ---
trade = trade.withColumn(
    "new_codes",
    F.when(
        F.col("diff") >= 36,
        F.array_repeat(F.lit("YYY"), 36)
    ).otherwise(
        F.concat(
            F.array_repeat(F.lit("YYY"), F.col("diff")),
            F.slice(F.col("raw_codes"), 1, F.lit(36) - F.col("diff"))
        )
    )
)

In [98]:
trade.select("raw_codes", "new_codes").show(3, truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|raw_codes                                                                                                                                                                           |new_codes                                                                                                                                                                           |
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------

In [99]:
# --- Step 5: Create 36 DPD columns ---
# element_at is 1-based in Spark → index 1 = new_codes[0] = DPD_M1

for i in range(1, 37):
    trade = trade.withColumn(
        f"DPD_M{i}",
        F.when(
            F.col("new_codes").isNull(),
            F.lit(-1)
        ).when(
            F.element_at(F.col("new_codes"), i).isin("XXX", "YYY"),
            F.lit(-1)
        ).otherwise(
            F.element_at(F.col("new_codes"), i).cast("int")
        )
    )

In [100]:
trade.select("user_id", "DPD_M1", "DPD_M2", "DPD_M3", "DPD_M15", "DPD_M20", "DPD_M26", "DPD_M36", "new_codes").show(10, truncate=True)

+-----------+------+------+------+-------+-------+-------+-------+--------------------+
|    user_id|DPD_M1|DPD_M2|DPD_M3|DPD_M15|DPD_M20|DPD_M26|DPD_M36|           new_codes|
+-----------+------+------+------+-------+-------+-------+-------+--------------------+
| 8589943531|    -1|    -1|    -1|     -1|     -1|     -1|      0|[YYY, YYY, YYY, Y...|
| 8589943531|    -1|    -1|    -1|     -1|     -1|     -1|      0|[YYY, YYY, YYY, Y...|
| 8589943531|    -1|    -1|    -1|      0|      0|      0|      0|[YYY, YYY, YYY, Y...|
|60129548715|    -1|    -1|    -1|     -1|     -1|     -1|     -1|[YYY, YYY, YYY, Y...|
|       8171|    -1|     0|     0|      0|     -1|     -1|     -1|[YYY, 000, 000, 0...|
|42949680004|    -1|     0|     0|     -1|     -1|     -1|     -1|[YYY, 000, 000, 0...|
|42949680004|    -1|    -1|    -1|     -1|     -1|     -1|      0|[YYY, YYY, YYY, Y...|
|42949680004|    -1|     0|     0|      0|      0|      0|      0|[YYY, 000, 000, 0...|
|60129542640|    -1|    -1|    -

In [101]:
# --- Step 6: Create 36 DPD Buckets columns---
# element_at is 1-based in Spark → index 1 = new_codes[0] = DPD_M1

for i in range(1, 37):
    code_col = F.element_at(F.col("new_codes"), i)

    trade = trade.withColumn( f"DPD_M{i}_bucket",
                F.when(F.col("new_codes").isNull(), F.lit(-1))
                 .when(code_col.isNull(), F.lit(-1))
                             
                # Special CIBIL codes
                 .when(code_col.isin("XXX", "YYY"), F.lit(-1))
                 .when(code_col == "901", F.lit(0))
                 .when(code_col == "902", F.lit(4))
                 .when(code_col.isin("903", "904"), F.lit(6))
                 .when(code_col == "905", F.lit(3))
                 .when(code_col == "000", F.lit(0))
                             
                 .when(code_col.cast("int").between(1, 29), F.lit(1))
                 .when(code_col.cast("int").between(30, 59), F.lit(2))
                 .when(code_col.cast("int").between(60, 89), F.lit(3))
                 .when(code_col.cast("int").between(90, 149), F.lit(4))
                 .when(code_col.cast("int").between(150, 179), F.lit(5))
                 .when(code_col.cast("int").between(180, 359), F.lit(6))
                 .when(code_col.cast("int") >= 360, F.lit(7))
                 .otherwise(F.lit(-1))
    )

In [102]:
trade.select("user_id", "DPD_M1_bucket", "DPD_M2_bucket", "DPD_M3_bucket", "DPD_M15_bucket", "DPD_M20_bucket", "DPD_M26_bucket", "DPD_M36_bucket", "new_codes").show(10, truncate=True)

+-----------+-------------+-------------+-------------+--------------+--------------+--------------+--------------+--------------------+
|    user_id|DPD_M1_bucket|DPD_M2_bucket|DPD_M3_bucket|DPD_M15_bucket|DPD_M20_bucket|DPD_M26_bucket|DPD_M36_bucket|           new_codes|
+-----------+-------------+-------------+-------------+--------------+--------------+--------------+--------------+--------------------+
| 8589943531|           -1|           -1|           -1|            -1|            -1|            -1|             0|[YYY, YYY, YYY, Y...|
| 8589943531|           -1|           -1|           -1|            -1|            -1|            -1|             0|[YYY, YYY, YYY, Y...|
| 8589943531|           -1|           -1|           -1|             0|             0|             0|             0|[YYY, YYY, YYY, Y...|
|60129548715|           -1|           -1|           -1|            -1|            -1|            -1|            -1|[YYY, YYY, YYY, Y...|
|       8171|           -1|            0|

In [103]:
trade.printSchema()

root
 |-- ACCOUNT_TYPE_CODE: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- ACCOUNT_TYPE: string (nullable = true)
 |-- REPORTING_MEMBER_NAME: string (nullable = true)
 |-- HIGHEST_CREDIT_OR_LOAN_AMOUNT: double (nullable = true)
 |-- OPEN_DATE: date (nullable = true)
 |-- REPORTING_DATE: date (nullable = true)
 |-- CLOSED_DATE: date (nullable = true)
 |-- LAST_PAYMENT_DATE: date (nullable = true)
 |-- INTEREST_RATE: double (nullable = true)
 |-- EMI_AMOUNT: double (nullable = true)
 |-- TENURE: string (nullable = true)
 |-- CURRENT_BALANCE: double (nullable = true)
 |-- AMOUNT_OVERDUE: double (nullable = true)
 |-- WRITTEN_OFF_AMOUNT: double (nullable = true)
 |-- WRITTEN_OFF_AMOUNT_PRINCIPAL: double (nullable = true)
 |-- IS_SUIT_FILED_OR_WILFUL_DEFAULT: decimal(38,0) (nullable = true)
 |-- IS_WRITTEN_OFF_OR_SETTLED: decimal(38,0) (nullable = true)
 |-- PAYMENT_HISTORY_START_DATE: date (nullable = true)
 |-- PAYMENT_HISTORY_END_DATE: date (nullable = true)
 |-- ACC

## Creating Features:

### Tradeline_Features

In [104]:
trade.show(1)

+-----------------+----------+-------------+---------------------+-----------------------------+----------+--------------+-----------+-----------------+-------------+----------+------+---------------+--------------+------------------+----------------------------+-------------------------------+-------------------------+--------------------------+------------------------+-------------------+------------+----------+---------------------+----------------+---------------+--------------------+----------+--------------+---------+-------+--------------------+----+--------------------+------+------+------+------+------+------+------+------+------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-----------

In [105]:
# Count null in each columns
null_counts = trade.select([F.count(F.when(F.col(c).isNull(),c)).alias(c) for c in trade.columns])
null_counts.show()

+-----------------+-------+------------+---------------------+-----------------------------+---------+--------------+-----------+-----------------+-------------+----------+------+---------------+--------------+------------------+----------------------------+-------------------------------+-------------------------+--------------------------+------------------------+-------------------+------------+----------+---------------------+----------------+---------------+-------------------+----------+--------------+---------+-------+---------+----+---------+------+------+------+------+------+------+------+------+------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+--------------+--------------+---------

In [106]:
null_counts.select("ACCOUNT_TYPE_CODE", "user_id", "ACCOUNT_TYPE", "REPORTING_MEMBER_NAME", "HIGHEST_CREDIT_OR_LOAN_AMOUNT", "OPEN_DATE", "REPORTING_DATE", "CLOSED_DATE", "LAST_PAYMENT_DATE", "INTEREST_RATE").show()

+-----------------+-------+------------+---------------------+-----------------------------+---------+--------------+-----------+-----------------+-------------+
|ACCOUNT_TYPE_CODE|user_id|ACCOUNT_TYPE|REPORTING_MEMBER_NAME|HIGHEST_CREDIT_OR_LOAN_AMOUNT|OPEN_DATE|REPORTING_DATE|CLOSED_DATE|LAST_PAYMENT_DATE|INTEREST_RATE|
+-----------------+-------+------------+---------------------+-----------------------------+---------+--------------+-----------+-----------------+-------------+
|                0|      0|           0|                    0|                            0|        0|             0|     271823|           129867|            0|
+-----------------+-------+------------+---------------------+-----------------------------+---------+--------------+-----------+-----------------+-------------+



In [107]:
null_counts.select("EMI_AMOUNT", "TENURE", "CURRENT_BALANCE", "AMOUNT_OVERDUE", "WRITTEN_OFF_AMOUNT", "WRITTEN_OFF_AMOUNT_PRINCIPAL", "IS_SUIT_FILED_OR_WILFUL_DEFAULT", "IS_WRITTEN_OFF_OR_SETTLED").show()

+----------+------+---------------+--------------+------------------+----------------------------+-------------------------------+-------------------------+
|EMI_AMOUNT|TENURE|CURRENT_BALANCE|AMOUNT_OVERDUE|WRITTEN_OFF_AMOUNT|WRITTEN_OFF_AMOUNT_PRINCIPAL|IS_SUIT_FILED_OR_WILFUL_DEFAULT|IS_WRITTEN_OFF_OR_SETTLED|
+----------+------+---------------+--------------+------------------+----------------------------+-------------------------------+-------------------------+
|         0|351650|              0|             0|                 0|                           0|                              0|                        0|
+----------+------+---------------+--------------+------------------+----------------------------+-------------------------------+-------------------------+



In [108]:
null_counts.select("PAYMENT_HISTORY_START_DATE", "PAYMENT_HISTORY_END_DATE", "ACCOUNT_HOLDER_TYPE", "CREDIT_LIMIT", "CASH_LIMIT", "ACTUAL_PAYMENT_AMOUNT", "COLLATERAL_VALUE", "COLLATERAL_TYPE").show()

+--------------------------+------------------------+-------------------+------------+----------+---------------------+----------------+---------------+
|PAYMENT_HISTORY_START_DATE|PAYMENT_HISTORY_END_DATE|ACCOUNT_HOLDER_TYPE|CREDIT_LIMIT|CASH_LIMIT|ACTUAL_PAYMENT_AMOUNT|COLLATERAL_VALUE|COLLATERAL_TYPE|
+--------------------------+------------------------+-------------------+------------+----------+---------------------+----------------+---------------+
|                         0|                       0|                  1|           0|         0|                    0|               0|         818637|
+--------------------------+------------------------+-------------------+------------+----------+---------------------+----------------+---------------+



In [109]:
null_counts.select("payment_history_str", "retro_date", "BROAD_CATEGORY", "REVOLVING", "SECURED", "raw_codes", "diff", "new_codes").show()

+-------------------+----------+--------------+---------+-------+---------+----+---------+
|payment_history_str|retro_date|BROAD_CATEGORY|REVOLVING|SECURED|raw_codes|diff|new_codes|
+-------------------+----------+--------------+---------+-------+---------+----+---------+
|               6523|         0|             0|        0|      0|     6523|   0|     6452|
+-------------------+----------+--------------+---------+-------+---------+----+---------+



In [110]:
# Columns that have null value(s) are: CLOSED_DATE, LAST_PAYMENT_DATE, TENURE, ACCOUNT_HOLDER_TYPE, COLLATERAL_TYPE, 
#                                      payment_history_str, raw_codes, new_codes.

In [111]:
# F.min_by(column_to_return, column_to_evaluate)

In [112]:
# For feature 12:
# The is_live flag tells us: "Is this trade currently open/active as of the retro_date?"

trade = trade.withColumn(
    "is_live",
    F.when(
        (F.col("OPEN_DATE") <= F.col("retro_date")) &
        ((F.col("CLOSED_DATE").isNull()) | (F.col("CLOSED_DATE") > F.col("retro_date"))),
        1
    ).otherwise(0)
)

In [113]:
# 127 - DPD_RECOVERY_RATE

dpd_cols_12 = ", ".join([f"DPD_M{i}" for i in range(1, 13)])

trade = trade.withColumn(
    "DPD_PEAK_12M",
    F.greatest(*[F.col(f"DPD_M{i}") for i in range(1, 13)])
)

In [114]:
# 128 - 129 DPD_SLOPE_6M, DPD_SLOPE_12M

# For slope6 
#   6      = n
#   21     = Σx
#   105    = nΣx² - (Σx)²

slope6 = (
    (
        6 * (
            1 * F.col("DPD_M1") +
            2 * F.col("DPD_M2") +
            3 * F.col("DPD_M3") +
            4 * F.col("DPD_M4") +
            5 * F.col("DPD_M5") +
            6 * F.col("DPD_M6")
        ) -
        21 * (
            F.col("DPD_M1") + F.col("DPD_M2") + F.col("DPD_M3") +
            F.col("DPD_M4") + F.col("DPD_M5") + F.col("DPD_M6")
        )
    ) / F.lit(105.0)
)

slope12 = (
    (
        12 * (
            1 * F.col("DPD_M1") +
            2 * F.col("DPD_M2") +
            3 * F.col("DPD_M3") +
            4 * F.col("DPD_M4") +
            5 * F.col("DPD_M5") +
            6 * F.col("DPD_M6") +
            7 * F.col("DPD_M7") +
            8 * F.col("DPD_M8") +
            9 * F.col("DPD_M9") +
            10 * F.col("DPD_M10") +
            11 * F.col("DPD_M11") +
            12 * F.col("DPD_M12")
        ) -
        78 * (
            F.col("DPD_M1") + F.col("DPD_M2") + F.col("DPD_M3") +
            F.col("DPD_M4") + F.col("DPD_M5") + F.col("DPD_M6") +
            F.col("DPD_M7") + F.col("DPD_M8") + F.col("DPD_M9") +
            F.col("DPD_M10") + F.col("DPD_M11") + F.col("DPD_M12")
        )
    ) / F.lit(1716.0)
)

trade = trade.withColumn(
    "DPD_SLOPE_6M_ROW",
    F.when(
        (F.col("DPD_M1") >= 0) & (F.col("DPD_M2") >= 0) & (F.col("DPD_M3") >= 0) &
        (F.col("DPD_M4") >= 0) & (F.col("DPD_M5") >= 0) & (F.col("DPD_M6") >= 0),
        slope6
    )
).withColumn(
    "DPD_SLOPE_12M_ROW",
    F.when(
        (F.col("DPD_M1") >= 0) & (F.col("DPD_M2") >= 0) & (F.col("DPD_M3") >= 0) &
        (F.col("DPD_M4") >= 0) & (F.col("DPD_M5") >= 0) & (F.col("DPD_M6") >= 0) &
        (F.col("DPD_M7") >= 0) & (F.col("DPD_M8") >= 0) & (F.col("DPD_M9") >= 0) &
        (F.col("DPD_M10") >= 0) & (F.col("DPD_M11") >= 0) & (F.col("DPD_M12") >= 0),
        slope12
    )
)

In [115]:
# 130: DPD_CURRENT_VS_HISTORY_RATIO
dpd_hist_36 = F.greatest(*[F.col(f"DPD_M{i}") for i in range(1, 37)])

trade = trade.withColumn(
    "DPD_CURRENT_VS_HISTORY_RATIO_ROW",
    F.when(
        (F.col("DPD_M1") >= 0) & (dpd_hist_36 > 0),
        F.col("DPD_M1") / dpd_hist_36
    )
)

In [116]:
# For 131 : NUM_CONSEC_MONTHS_ZERO_DPD (see Below)

streak_candidates = []

for length in range(1, 37):
    for start in range(1, 37 - length + 1):
        cond = F.lit(True)
        for i in range(start, start + length):
            cond = cond & (F.col(f"DPD_M{i}") == 0)   # fixed

        streak_candidates.append(
            F.when(cond, F.lit(length)).otherwise(F.lit(0))
        )

trade = trade.withColumn(
    "NUM_CONSEC_MONTHS_ZERO_DPD_ROW",
    F.greatest(*streak_candidates)
)

In [117]:
trade_features = trade.groupBy("user_id").agg(
    F.min_by("BROAD_CATEGORY", "OPEN_DATE").alias("FIRST_PRODUCT"), #1
    F.max_by("BROAD_CATEGORY", "OPEN_DATE").alias("LATEST_PRODUCT"), #2
    F.max(F.months_between(F.col("retro_date"), F.col("OPEN_DATE"))).alias("MAX_BUR_VINTAGE"), #Maximum bureau vintage in months (months_since_open of the oldest trade).
    F.count("*").alias("TOTAL_TRADES"), #4
    F.sum(F.when(F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -1), 1).otherwise(0)).alias("TOTAL_TRADES_1_MN"),  #5
    F.sum(F.when(F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -3), 1).otherwise(0)).alias("TOTAL_TRADES_3_MN"), #6
    F.sum(F.when(F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -6), 1).otherwise(0)).alias("TOTAL_TRADES_6_MN"), #7
    F.sum(F.when(F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -12), 1).otherwise(0)).alias("TOTAL_TRADES_12_MN"), #8
    F.sum(F.when(F.col("BROAD_CATEGORY") == "HL", 1).otherwise(0)).alias("TOTAL_HL_TRADES"), #9
    F.sum(F.when(F.col("BROAD_CATEGORY") == "GL", 1).otherwise(0)).alias("TOTAL_GL_TRADES"), #10
    F.sum(F.when(F.col("BROAD_CATEGORY") == "PL", 1).otherwise(0)).alias("TOTAL_PL_TRADES"), #11
    F.sum(F.when((F.col("BROAD_CATEGORY") == "PL") & (F.col("is_live") == 1), 1).otherwise(0)).alias("TOTAL_LIVE_PL_TRADES"), #12
    F.sum(F.when((F.col("BROAD_CATEGORY") == "PL") & (F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -1)), 1).otherwise(0)).alias("TOTAL_PL_TRADES_1_MN"), #13
    F.sum(F.when((F.col("BROAD_CATEGORY") == "PL") & (F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -3)), 1).otherwise(0)).alias("TOTAL_PL_TRADES_3_MN"), #14
    F.sum(F.when((F.col("BROAD_CATEGORY") == "PL") & (F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -6)), 1).otherwise(0)).alias("TOTAL_PL_TRADES_6_MN"), #15
    F.sum(F.when((F.col("BROAD_CATEGORY") == "PL") & (F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -12)), 1).otherwise(0)).alias("TOTAL_PL_TRADES_12_MN"), #16
    F.sum(F.when(F.col("BROAD_CATEGORY") == "CC", 1).otherwise(0)).alias("TOTAL_CC_TRADES"), #17
    F.sum(F.when((F.col("BROAD_CATEGORY") == "CC") & (F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -1)), 1).otherwise(0)).alias("TOTAL_CC_TRADES_1_MN"), #18
    F.sum(F.when((F.col("BROAD_CATEGORY") == "CC") & (F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -3)), 1).otherwise(0)).alias("TOTAL_CC_TRADES_3_MN"), #19
    F.sum(F.when((F.col("BROAD_CATEGORY") == "CC") & (F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -6)), 1).otherwise(0)).alias("TOTAL_CC_TRADES_6_MN"), #20
    F.sum(F.when((F.col("BROAD_CATEGORY") == "CC") & (F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -12)), 1).otherwise(0)).alias("TOTAL_CC_TRADES_12_MN"), #21
    F.sum(F.when(F.col("SECURED") == 1, 1).otherwise(0)).alias("TOTAL_SEC_TRADES"), #22
    F.sum(F.when(F.col("SECURED") == 0, 1).otherwise(0)).alias("TOTAL_UNSEC_TRADES"), #23
    F.sum(F.when((F.col("SECURED") == 0) & (F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -1)), 1).otherwise(0)).alias("TOTAL_UNSEC_TRADES_1_MN"), #24
    F.sum(F.when((F.col("SECURED") == 0) & (F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -3)), 1).otherwise(0)).alias("TOTAL_UNSEC_TRADES_3_MN"), #25
    F.sum(F.when((F.col("SECURED") == 0) & (F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -6)), 1).otherwise(0)).alias("TOTAL_UNSEC_TRADES_6_MN"), #26
    F.sum(F.when((F.col("SECURED") == 0) & (F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -12)), 1).otherwise(0)).alias("TOTAL_UNSEC_TRADES_12_MN"), #27
    F.sum(F.when(F.col("REVOLVING") == 0, 1).otherwise(0)).alias("TOTAL_INSTALLMENT_TRADES"), #28

    # LiveWindow (29 - 31)
    F.sum(F.when(
        (F.col("DPD_M1") != -1) | (F.col("DPD_M2") != -1) | (F.col("DPD_M3") != -1),
        1
    ).otherwise(0)).alias("TOTAL_LIVE_TRADES_M0_M2"),

    F.sum(F.when(
        ((F.col("DPD_M1") != -1) | (F.col("DPD_M2") != -1) | (F.col("DPD_M3") != -1)) &
        (F.col("SECURED") == 0),
        1
    ).otherwise(0)).alias("TOTAL_LIVE_UNSEC_TRADES_M0_M2"),

    F.sum(F.when(
        ((F.col("DPD_M1") != -1) | (F.col("DPD_M2") != -1) | (F.col("DPD_M3") != -1)) &
        (F.col("REVOLVING") == 0),
        1
    ).otherwise(0)).alias("TOTAL_LIVE_INSTALLMENT_TRADES_M0_M2"),
    
    
    # Balance (32-38)
    F.sum(F.coalesce(F.col("CURRENT_BALANCE"), F.lit(0))).alias("TOTAL_BALANCE"), #32
    F.sum(F.when(F.col("BROAD_CATEGORY") == "PL", F.coalesce(F.col("CURRENT_BALANCE"), F.lit(0))).otherwise(0)).alias("TOTAL_PL_BALANCE"), #33 (If "CURRENT_BALANCE"=NULL, then takes its value=0)
    F.sum(F.when(F.col("BROAD_CATEGORY") == "BL", F.coalesce(F.col("CURRENT_BALANCE"), F.lit(0))).otherwise(0)).alias("TOTAL_BL_BALANCE"), #34
    F.sum(F.when(F.col("BROAD_CATEGORY") == "CD", F.coalesce(F.col("CURRENT_BALANCE"), F.lit(0))).otherwise(0)).alias("TOTAL_CD_BALANCE"), #35
    F.sum(F.when(F.col("BROAD_CATEGORY") == "AL", F.coalesce(F.col("CURRENT_BALANCE"), F.lit(0))).otherwise(0)).alias("TOTAL_AL_BALANCE"), #36
    F.sum(F.when(F.col("BROAD_CATEGORY") == "HL", F.coalesce(F.col("CURRENT_BALANCE"), F.lit(0))).otherwise(0)).alias("TOTAL_HL_BALANCE"), #37
    F.sum(F.when(F.col("BROAD_CATEGORY") == "CC", F.coalesce(F.col("CURRENT_BALANCE"), F.lit(0))).otherwise(0)).alias("TOTAL_CC_BALANCE"), #38

    # Sanction Amount (39-45)
    F.sum(F.coalesce(F.col("HIGHEST_CREDIT_OR_LOAN_AMOUNT"), F.lit(0))).alias("TOTAL_SANC_AMT"), #39
    F.sum(F.when(F.col("BROAD_CATEGORY") == "PL", F.coalesce(F.col("HIGHEST_CREDIT_OR_LOAN_AMOUNT"), F.lit(0))).otherwise(0)).alias("TOTAL_PL_SANC_AMT"), #40
    F.sum(F.when(F.col("BROAD_CATEGORY") == "BL", F.coalesce(F.col("HIGHEST_CREDIT_OR_LOAN_AMOUNT"), F.lit(0))).otherwise(0)).alias("TOTAL_BL_SANC_AMT"), #41
    F.sum(F.when(F.col("BROAD_CATEGORY") == "CD", F.coalesce(F.col("HIGHEST_CREDIT_OR_LOAN_AMOUNT"), F.lit(0))).otherwise(0)).alias("TOTAL_CD_SANC_AMT"), #42
    F.sum(F.when(F.col("BROAD_CATEGORY") == "AL", F.coalesce(F.col("HIGHEST_CREDIT_OR_LOAN_AMOUNT"), F.lit(0))).otherwise(0)).alias("TOTAL_AL_SANC_AMT"), #43
    F.sum(F.when(F.col("BROAD_CATEGORY") == "HL", F.coalesce(F.col("HIGHEST_CREDIT_OR_LOAN_AMOUNT"), F.lit(0))).otherwise(0)).alias("TOTAL_HL_SANC_AMT"), #44
    F.sum(F.when(F.col("BROAD_CATEGORY") == "CC", F.coalesce(F.col("HIGHEST_CREDIT_OR_LOAN_AMOUNT"), F.lit(0))).otherwise(0)).alias("TOTAL_CC_SANC_AMT"), #45

    # Past Due (46-48)
    F.sum(F.coalesce(F.col("AMOUNT_OVERDUE"), F.lit(0))).alias("TOTAL_AMT_PAST_DUE"), #46
    F.sum(F.when(
        (F.col("IS_WRITTEN_OFF_OR_SETTLED").cast("int") == 1) | 
        (F.col("IS_SUIT_FILED_OR_WILFUL_DEFAULT").cast("int") == 1), 
        1
    ).otherwise(0)).alias("TOTAL_WO_SF_TRADES"),   #47
    
    F.sum(F.when(
        ((F.col("IS_WRITTEN_OFF_OR_SETTLED").cast("int") == 1) | 
         (F.col("IS_SUIT_FILED_OR_WILFUL_DEFAULT").cast("int") == 1)) &
        (F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -36)),
        1
    ).otherwise(0)).alias("TOTAL_WO_SF_36O_TRADES"),  #48

    # Utilization (49-50)
    # MACRO_UTIL: sum(CC balance) / sum(CC credit limit)
    F.when(
        F.sum(F.when(F.col("BROAD_CATEGORY") == "CC", F.coalesce(F.col("CREDIT_LIMIT"), F.lit(0))).otherwise(0)) > 0,
        F.sum(F.when(F.col("BROAD_CATEGORY") == "CC", F.coalesce(F.col("CURRENT_BALANCE"), F.lit(0))).otherwise(0)) /
        F.sum(F.when(F.col("BROAD_CATEGORY") == "CC", F.coalesce(F.col("CREDIT_LIMIT"), F.lit(0))).otherwise(0))
    ).otherwise(-1).alias("MACRO_UTIL"),

    # MICRO_UTIL: avg(per-trade balance/limit) for CC trades with limit > 0
    F.when(
        F.sum(F.when((F.col("BROAD_CATEGORY") == "CC") & (F.col("CREDIT_LIMIT") > 0), 1).otherwise(0)) > 0,
        F.sum(F.when((F.col("BROAD_CATEGORY") == "CC") & (F.col("CREDIT_LIMIT") > 0), 
                 F.coalesce(F.col("CURRENT_BALANCE"), F.lit(0)) / F.col("CREDIT_LIMIT")).otherwise(0)) /
        F.sum(F.when((F.col("BROAD_CATEGORY") == "CC") & (F.col("CREDIT_LIMIT") > 0), 1).otherwise(0))
    ).otherwise(-1).alias("MICRO_UTIL"),


    # DPD_BucketCount (73-75)
    
#Step-1: (See below cells for Step-2 & -3)
#       Creating Month flags for DPD >= 0 (H0003, H0012, H0036) [See Below Cell]
    *[
        F.max(
            F.when(F.col(f"DPD_M{i}") >= 0, 1).otherwise(0)
        ).alias(f"flag_M{i}_has_dpd_ge_eq_0")
        for i in range(1, 37)
    ],

    # Creating Month flags for DPD >= 1 (H0103, H0112, H0136)
    *[
        F.max(
            F.when(F.col(f"DPD_M{i}") >= 1, 1).otherwise(0)
        ).alias(f"flag_M{i}_has_dpd_ge_eq_1")
        for i in range(1, 37)
    ],

    ## Creating Month flags for DPD >= 30 (H0203, H0212, H0236)
    *[
        F.max(
            F.when(F.col(f"DPD_M{i}") >= 30, 1).otherwise(0)
        ).alias(f"flag_M{i}_has_dpd_ge_eq_30")
        for i in range(1, 37)
    ],

    # Month flags for DPD >= 60 (H0303, H0312, H0336)
    *[
        F.max(
            F.when(F.col(f"DPD_M{i}") >= 60, 1).otherwise(0)
        ).alias(f"flag_M{i}_has_dpd_ge_eq_60")
        for i in range(1, 37)
    ],
    
    # Month flags for DPD >= 90 (H0403, H0412, H0436)
    *[
        F.max(
            F.when(F.col(f"DPD_M{i}") >= 90, 1).otherwise(0)
        ).alias(f"flag_M{i}_has_dpd_ge_eq_90")
        for i in range(1, 37)
    ],
    
    # Month flags for DPD >= 180 (H0503, H0512, H0536)
    *[
        F.max(
            F.when(F.col(f"DPD_M{i}") >= 180, 1).otherwise(0)
        ).alias(f"flag_M{i}_has_dpd_ge_eq_180")
        for i in range(1, 37)
    ],

    # Month flags for DPD >= 30 on PL trades
    *[
        F.max(
            F.when(
                (F.col(f"DPD_M{i}") >= 30) & (F.col("BROAD_CATEGORY") == "PL"),
                1
            ).otherwise(0)
        ).alias(f"flag_M{i}_has_dpd_ge_eq_30_pl")
        for i in range(1, 37)
    ],

    # Month flags for PL trades, DPD >= 90 (H1403, H1412, H1436)
    *[
        F.max(
            F.when(
                (F.col("BROAD_CATEGORY") == "PL") & (F.col(f"DPD_M{i}") >= 90),
                1
            ).otherwise(0)
        ).alias(f"flag_M{i}_has_dpd_ge_eq_90_pl")
        for i in range(1, 37)
    ],

    # Month flags for PL trades, DPD >= 180 (H1503, H1512, H1536)
    *[
        F.max(
            F.when(
                (F.col("BROAD_CATEGORY") == "PL") & (F.col(f"DPD_M{i}") >= 180),
                1
            ).otherwise(0)
        ).alias(f"flag_M{i}_has_dpd_ge_eq_180_pl")
        for i in range(1, 37)
    ],

    # Month flags for CC trades, DPD >= 30 (H2203, H2212, H2236)
    *[
        F.max(
            F.when(
                (F.col("BROAD_CATEGORY") == "CC") & (F.col(f"DPD_M{i}") >= 30),
                1
            ).otherwise(0)
        ).alias(f"flag_M{i}_has_dpd_ge_eq_30_cc")
        for i in range(1, 37)
    ],

    # Month flags for CC trades, DPD >= 90 (H2403, H2412, H2436)
    *[
        F.max(
            F.when(
                (F.col("BROAD_CATEGORY") == "CC") & (F.col(f"DPD_M{i}") >= 90),
                1
            ).otherwise(0)
        ).alias(f"flag_M{i}_has_dpd_ge_eq_90_cc")
        for i in range(1, 37)
    ],

    # Month flags for CC trades, DPD >= 180 (H2503, H2512, H2536)
    *[
        F.max(
            F.when(
                (F.col("BROAD_CATEGORY") == "CC") & (F.col(f"DPD_M{i}") >= 180),
                1
            ).otherwise(0)
        ).alias(f"flag_M{i}_has_dpd_ge_eq_180_cc")
        for i in range(1, 37)
    ],

    # Month flags for unsecured trades, DPD >= 30 (H3203, H3212, H3236)
    *[
        F.max(
            F.when(
                (F.col("SECURED") == 0) & (F.col(f"DPD_M{i}") >= 30),
                1
            ).otherwise(0)
        ).alias(f"flag_M{i}_has_dpd_ge_eq_30_unsec")
        for i in range(1, 37)
    ],

    # Month flags for unsecured trades, DPD >= 90 (H3403, H3412, H3436)
    *[
        F.max(
            F.when(
                (F.col("SECURED") == 0) & (F.col(f"DPD_M{i}") >= 90),
                1
            ).otherwise(0)
        ).alias(f"flag_M{i}_has_dpd_ge_eq_90_unsec")
        for i in range(1, 37)
    ],

    # Month flags for unsecured trades, DPD >= 180 (H3503, H3512, H3536)
    *[
        F.max(
            F.when(
                (F.col("SECURED") == 0) & (F.col(f"DPD_M{i}") >= 180),
                1
            ).otherwise(0)
        ).alias(f"flag_M{i}_has_dpd_ge_eq_180_unsec")
        for i in range(1, 37)
    ],

    # H4103 - H4136
    # H4103 – max DPD in months 1‑3
    F.max(F.greatest(F.col("DPD_M1"), F.col("DPD_M2"), F.col("DPD_M3"))).alias("H4103"),
    
    # H4112 – max DPD in months 1‑12
    F.max(F.greatest(*[F.col(f"DPD_M{i}") for i in range(1, 13)])).alias("H4112"),
    
    # H4136 – max DPD in months 1‑36
    F.max(F.greatest(*[F.col(f"DPD_M{i}") for i in range(1, 37)])).alias("H4136"),

    # DPD_MaxLevel (121-123) - PL trades
    F.greatest(
        *[F.max(F.when(F.col("BROAD_CATEGORY") == "PL", F.col(f"DPD_M{i}"))) for i in range(1, 4)]
    ).alias("H4203"),
    
    F.greatest(
        *[F.max(F.when(F.col("BROAD_CATEGORY") == "PL", F.col(f"DPD_M{i}"))) for i in range(1, 13)]
    ).alias("H4212"),
    
    F.greatest(
        *[F.max(F.when(F.col("BROAD_CATEGORY") == "PL", F.col(f"DPD_M{i}"))) for i in range(1, 37)]
    ).alias("H4236"),

    # DPD_MaxLevel (124-126) - unsecured trades
    F.greatest(
        *[F.max(F.when(F.col("SECURED") == 0, F.col(f"DPD_M{i}"))) for i in range(1, 4)]
    ).alias("H4303"),
    
    F.greatest(
        *[F.max(F.when(F.col("SECURED") == 0, F.col(f"DPD_M{i}"))) for i in range(1, 13)]
    ).alias("H4312"),
    
    F.greatest(
        *[F.max(F.when(F.col("SECURED") == 0, F.col(f"DPD_M{i}"))) for i in range(1, 37)]
    ).alias("H4336"),



    # 127 - DPD_RECOVERY_RATE
    F.max(
        F.expr(f"""
            CASE
                WHEN DPD_PEAK_12M > 0
                 AND array_position(array({dpd_cols_12}), DPD_PEAK_12M) > 1
                THEN (DPD_PEAK_12M - DPD_M1) /
                     cast(array_position(array({dpd_cols_12}), DPD_PEAK_12M) as double)
                ELSE NULL
            END
        """)
    ).alias("DPD_RECOVERY_RATE"),


    # 128 - 129: DPD_SLOPE_6M, DPD_SLOPE_12M (See Above)
    F.avg("DPD_SLOPE_6M_ROW").alias("DPD_SLOPE_6M"),
    F.avg("DPD_SLOPE_12M_ROW").alias("DPD_SLOPE_12M"),

    #130 (See Above)
    F.max("DPD_CURRENT_VS_HISTORY_RATIO_ROW").alias("DPD_CURRENT_VS_HISTORY_RATIO"),

    #131 (See Above)
    F.max("NUM_CONSEC_MONTHS_ZERO_DPD_ROW").alias("NUM_CONSEC_MONTHS_ZERO_DPD"),

    #132: ACCOUNT_CLOSURE_RATE_6M = Accounts closed in last 6 months / accounts that were active 6 months ago.
    F.when(
        F.sum(
            F.when(
                (F.col("OPEN_DATE") <= F.add_months(F.col("retro_date"), -6)) &
                (
                    F.col("CLOSED_DATE").isNull() |
                    (F.col("CLOSED_DATE") > F.add_months(F.col("retro_date"), -6))
                ),
                1
            ).otherwise(0)
        ) > 0,
        F.sum(
            F.when(
                (F.col("OPEN_DATE") <= F.add_months(F.col("retro_date"), -6)) &
                (F.col("CLOSED_DATE").isNotNull()) &
                (F.col("CLOSED_DATE") > F.add_months(F.col("retro_date"), -6)) &
                (F.col("CLOSED_DATE") <= F.col("retro_date")),
                1
            ).otherwise(0)
        ) /
        F.sum(
            F.when(
                (F.col("OPEN_DATE") <= F.add_months(F.col("retro_date"), -6)) &
                (
                    F.col("CLOSED_DATE").isNull() |
                    (F.col("CLOSED_DATE") > F.add_months(F.col("retro_date"), -6))
                ),
                1
            ).otherwise(0)
        )
    ).otherwise(-1).alias("ACCOUNT_CLOSURE_RATE_6M"),

    # 133 - PCT_NEW_UNSECURED_LOANS_6M = New unsecured accounts in last 6 months / total new accounts in last 6 months.
    F.when(
        F.sum(
            F.when(
                F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -6),
                1
            ).otherwise(0)
        ) > 0,
        F.sum(
            F.when(
                (F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -6)) &
                (F.col("SECURED") == 0),
                1
            ).otherwise(0)
        ) /
        F.sum(
            F.when(
                F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -6),
                1
            ).otherwise(0)
        )
    ).otherwise(F.lit(-1)).alias("PCT_NEW_UNSECURED_LOANS_6M"),

    # 134 - PCT_NEW_SECURED_LOANS_6M = New secured accounts in last 6 months / total new accounts in last 6 months.
    F.when(
        F.sum(
            F.when(
                F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -6),
                1
            ).otherwise(0)
        ) > 0,
        F.sum(
            F.when(
                (F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -6)) &
                (F.col("SECURED") == 1),
                1
            ).otherwise(0)
        ) /
        F.sum(
            F.when(
                F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -6),
                1
            ).otherwise(0)
        )
    ).otherwise(F.lit(-1)).alias("PCT_NEW_SECURED_LOANS_6M")
    
)

In [118]:
# Step-2: Compute H0003,..., using the temporary month flags

trade_features = trade_features.select(
    "*",

    # (H0003, H0012, H0036)
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_0" for i in range(1, 4)])).alias("H0003"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_0" for i in range(1, 13)])).alias("H0012"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_0" for i in range(1, 37)])).alias("H0036"),

    # (H0103, H0112, H0136)
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_1" for i in range(1, 4)])).alias("H0103"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_1" for i in range(1, 13)])).alias("H0112"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_1" for i in range(1, 37)])).alias("H0136"),

    # (H0203, H0212, H0236)
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_30" for i in range(1, 4)])).alias("H0203"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_30" for i in range(1, 13)])).alias("H0212"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_30" for i in range(1, 37)])).alias("H0236"),

    # (H0303, H0312, H0336)
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_60" for i in range(1, 4)])).alias("H0303"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_60" for i in range(1, 13)])).alias("H0312"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_60" for i in range(1, 37)])).alias("H0336"),

    # (H0403, H0412, H0436)
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_90" for i in range(1, 4)])).alias("H0403"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_90" for i in range(1, 13)])).alias("H0412"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_90" for i in range(1, 37)])).alias("H0436"),

    # (H0503, H0512, H0536)
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_180" for i in range(1, 4)])).alias("H0503"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_180" for i in range(1, 13)])).alias("H0512"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_180" for i in range(1, 37)])).alias("H0536"),

    # (H1203, H1212, H1236)
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_30_pl" for i in range(1, 4)])).alias("H1203"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_30_pl" for i in range(1, 13)])).alias("H1212"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_30_pl" for i in range(1, 37)])).alias("H1236"),

    # (H1403, H1412, H1436)
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_90_pl" for i in range(1, 4)])).alias("H1403"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_90_pl" for i in range(1, 13)])).alias("H1412"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_90_pl" for i in range(1, 37)])).alias("H1436"),

    # (H1503, H1512, H1536)
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_180_pl" for i in range(1, 4)])).alias("H1503"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_180_pl" for i in range(1, 13)])).alias("H1512"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_180_pl" for i in range(1, 37)])).alias("H1536"),

    # (H2203, H2212, H2236)
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_30_cc" for i in range(1, 4)])).alias("H2203"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_30_cc" for i in range(1, 13)])).alias("H2212"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_30_cc" for i in range(1, 37)])).alias("H2236"),

    # (H2403, H2412, H2436)
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_90_cc" for i in range(1, 4)])).alias("H2403"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_90_cc" for i in range(1, 13)])).alias("H2412"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_90_cc" for i in range(1, 37)])).alias("H2436"),

    # (H2503, H2512, H2536)
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_180_cc" for i in range(1, 4)])).alias("H2503"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_180_cc" for i in range(1, 13)])).alias("H2512"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_180_cc" for i in range(1, 37)])).alias("H2536"),

    # (H3203, H3212, H3236)
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_30_unsec" for i in range(1, 4)])).alias("H3203"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_30_unsec" for i in range(1, 13)])).alias("H3212"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_30_unsec" for i in range(1, 37)])).alias("H3236"),

    # (H3403, H3412, H3436)
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_90_unsec" for i in range(1, 4)])).alias("H3403"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_90_unsec" for i in range(1, 13)])).alias("H3412"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_90_unsec" for i in range(1, 37)])).alias("H3436"),

    # (H3503, H3512, H3536)
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_180_unsec" for i in range(1, 4)])).alias("H3503"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_180_unsec" for i in range(1, 13)])).alias("H3512"),
    F.expr(" + ".join([f"flag_M{i}_has_dpd_ge_eq_180_unsec" for i in range(1, 37)])).alias("H3536")
)

In [119]:
# Step-3: Drop temporary columns

trade_features = trade_features.drop(
    *[f"flag_M{i}_has_dpd_ge_eq_0"   for i in range(1, 37)],
    *[f"flag_M{i}_has_dpd_ge_eq_1"   for i in range(1, 37)],
    *[f"flag_M{i}_has_dpd_ge_eq_30"  for i in range(1, 37)],
    *[f"flag_M{i}_has_dpd_ge_eq_60"  for i in range(1, 37)],
    *[f"flag_M{i}_has_dpd_ge_eq_90"  for i in range(1, 37)],
    *[f"flag_M{i}_has_dpd_ge_eq_180" for i in range(1, 37)],
    
    *[f"flag_M{i}_has_dpd_ge_eq_30_pl" for i in range(1, 37)],
    *[f"flag_M{i}_has_dpd_ge_eq_90_pl" for i in range(1, 37)],
    *[f"flag_M{i}_has_dpd_ge_eq_180_pl" for i in range(1, 37)],

    *[f"flag_M{i}_has_dpd_ge_eq_30_cc" for i in range(1, 37)],
    *[f"flag_M{i}_has_dpd_ge_eq_90_cc" for i in range(1, 37)],
    *[f"flag_M{i}_has_dpd_ge_eq_180_cc" for i in range(1, 37)],

    *[f"flag_M{i}_has_dpd_ge_eq_30_unsec" for i in range(1, 37)],
    *[f"flag_M{i}_has_dpd_ge_eq_90_unsec" for i in range(1, 37)],
    *[f"flag_M{i}_has_dpd_ge_eq_180_unsec" for i in range(1, 37)]
)

In [120]:
trade.filter(F.col("COLLATERAL_VALUE").isNull()).count()

0

In [121]:
len(trade_features.columns)

113

In [122]:
# OpenGap Features: 51-56

In [123]:
# Users with <2 trades produce Null

# Step 1: Compute lag gap per customer ordered by OPEN_DATE
w = Window.partitionBy("user_id").orderBy(F.asc("OPEN_DATE"))

trade = trade.withColumn(
    "gap_days",
    F.datediff(F.col("OPEN_DATE"), F.lag("OPEN_DATE", 1).over(w))
)

months = [6, 12, 36]

# Step 2: Loops unpacked directly inside .agg()
opengap_features = (
    trade.groupBy("user_id").agg(
        
        # ── AVG features (51 - 53) — loop unpacked inline ──────────────────────────────
        *[
            F.avg(
                F.when(
                    (F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -mn)) &
                    F.col("gap_days").isNotNull(),
                    F.col("gap_days")
                )
            ).alias(f"AVG_DAYS_SO_IN_LAST_{mn}_MN")
            for mn in months
        ],

        # ── MEDIAN features (54 - 56) — loop unpacked inline ──────────────────────────
        *[
            F.percentile_approx(
                F.when(
                    (F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -mn)) &
                    F.col("gap_days").isNotNull(),
                    F.col("gap_days")
                ), 0.5
            ).alias(f"MEDIAN_DAYS_SO_IN_LAST_{mn}_MN")
            for mn in months
        ],

        # ── Unsecured trades AVG (57 - 59) ──────────────────────────────────────────
        *[
            F.avg(
                F.when(
                    (F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -mn)) &
                    (F.col("gap_days").isNotNull()) &
                    (F.col("SECURED")==0),
                    F.col("gap_days")
                )
            ).alias(f"AVG_DAYS_SO_UNSECURED_IN_LAST_{mn}_MN")
            for mn in months
        ],

         # ── Unsecured trades MEDIAN  (60 - 62) ──────────────────────────────────────────
        *[
            F.percentile_approx(
                F.when(
                    (F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -mn)) &
                    F.col("gap_days").isNotNull() &
                    (F.col("SECURED") == 0),
                    F.col("gap_days")
                ), 0.5
            ).alias(f"MEDIAN_DAYS_SO_UNSECURED_IN_LAST_{mn}_MN")
            for mn in months
        ],

        # ── AVG PL trades (63-65) ─────────────────────────────────────────────
        *[
            F.avg(
                F.when(
                    (F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -mn)) &
                    F.col("gap_days").isNotNull() &
                    (F.col("BROAD_CATEGORY") == "PL"),
                    F.col("gap_days")
                )
            ).alias(f"AVG_DAYS_SO_PL_IN_LAST_{mn}_MN")
            for mn in months
        ],

        # ── MEDIAN PL trades (66-68) ──────────────────────────────────────────
        *[
            F.percentile_approx(
                F.when(
                    (F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -mn)) &
                    F.col("gap_days").isNotNull() &
                    (F.col("BROAD_CATEGORY") == "PL"),
                    F.col("gap_days")
                ), 0.5
            ).alias(f"MEDIAN_DAYS_SO_PL_IN_LAST_{mn}_MN")
            for mn in months
        ]

    )
)

In [124]:
# Step 3: Fill null → -1
opengap_cols = (
    [f"AVG_DAYS_SO_IN_LAST_{mn}_MN"             for mn in months] +
    [f"MEDIAN_DAYS_SO_IN_LAST_{mn}_MN"           for mn in months] +
    [f"AVG_DAYS_SO_UNSECURED_IN_LAST_{mn}_MN"    for mn in months] +
    [f"MEDIAN_DAYS_SO_UNSECURED_IN_LAST_{mn}_MN" for mn in months] +
    [f"AVG_DAYS_SO_PL_IN_LAST_{mn}_MN"           for mn in months] +
    [f"MEDIAN_DAYS_SO_PL_IN_LAST_{mn}_MN"        for mn in months]
)

opengap_features = opengap_features.fillna(-1, subset=opengap_cols)

In [125]:
opengap_features.show(5)

+-------+------------------------+-------------------------+-------------------------+---------------------------+----------------------------+----------------------------+----------------------------------+-----------------------------------+-----------------------------------+-------------------------------------+--------------------------------------+--------------------------------------+---------------------------+----------------------------+----------------------------+------------------------------+-------------------------------+-------------------------------+
|user_id|AVG_DAYS_SO_IN_LAST_6_MN|AVG_DAYS_SO_IN_LAST_12_MN|AVG_DAYS_SO_IN_LAST_36_MN|MEDIAN_DAYS_SO_IN_LAST_6_MN|MEDIAN_DAYS_SO_IN_LAST_12_MN|MEDIAN_DAYS_SO_IN_LAST_36_MN|AVG_DAYS_SO_UNSECURED_IN_LAST_6_MN|AVG_DAYS_SO_UNSECURED_IN_LAST_12_MN|AVG_DAYS_SO_UNSECURED_IN_LAST_36_MN|MEDIAN_DAYS_SO_UNSECURED_IN_LAST_6_MN|MEDIAN_DAYS_SO_UNSECURED_IN_LAST_12_MN|MEDIAN_DAYS_SO_UNSECURED_IN_LAST_36_MN|AVG_DAYS_SO_PL_IN_LAST_6_MN|

In [126]:
trade_features = trade_features.join(opengap_features, on="user_id", how="left")

In [127]:
# 69 - 72

In [128]:
# Age in months of each trade at retro_date
trade = trade.withColumn(
    "vintage_months",
    F.months_between(F.col("retro_date"), F.col("OPEN_DATE"))
)

# Keep only trades opened in the last 36 months
trade_36 = trade.filter(
    (F.col("OPEN_DATE") >= F.add_months(F.col("retro_date"), -36)) &
    F.col("vintage_months").isNotNull()
)

vintage_features = (
    trade_36.groupBy("user_id").agg(
        F.avg(F.when(F.col("BROAD_CATEGORY") == "PL", F.col("vintage_months"))).alias("AVG_PL_BUR_VINTAGE"),
        F.avg(F.when(F.col("BROAD_CATEGORY") == "BL", F.col("vintage_months"))).alias("AVG_BL_BUR_VINTAGE"),
        F.avg(F.when(F.col("BROAD_CATEGORY") == "CC", F.col("vintage_months"))).alias("AVG_CC_BUR_VINTAGE"),
        F.avg(F.when(F.col("BROAD_CATEGORY") == "CD", F.col("vintage_months"))).alias("AVG_CD_BUR_VINTAGE"),
    )
)

vintage_features = vintage_features.fillna(
    -1,
    subset=["AVG_PL_BUR_VINTAGE", "AVG_BL_BUR_VINTAGE", "AVG_CC_BUR_VINTAGE", "AVG_CD_BUR_VINTAGE"]
)

In [129]:
vintage_features.show()

+-----------+------------------+------------------+------------------+------------------+
|    user_id|AVG_PL_BUR_VINTAGE|AVG_BL_BUR_VINTAGE|AVG_CC_BUR_VINTAGE|AVG_CD_BUR_VINTAGE|
+-----------+------------------+------------------+------------------+------------------+
|85899346347| 8.243176178461539|              -1.0|              -1.0|              -1.0|
|68719478199|       22.77419355|              -1.0|              -1.0|              -1.0|
|51539617617|       28.90322581|              -1.0|11.569892473333333|        17.5483871|
|       5385|       6.092741935|              -1.0|              -1.0|               6.0|
|34359746776|              -1.0|              -1.0|              -1.0|              -1.0|
|       2509|21.295698923333333|              -1.0|              -1.0|              -1.0|
|34359749525|17.610513740000002|              -1.0|              -1.0|12.673387098749998|
| 8589937582|15.342549923333335|11.978494623333333|              -1.0|              -1.0|
|257698108

In [130]:
trade_features = trade_features.join(vintage_features, on="user_id", how="left")

In [131]:
# Fill default Values

trade_features = trade_features.fillna({

    # Vintage (1-3)
    "FIRST_PRODUCT": "Other",
    "LATEST_PRODUCT": "Other",
    "MAX_BUR_VINTAGE": -1,

    # Counts (4-31)
    "TOTAL_TRADES": 0,
    "TOTAL_TRADES_1_MN": 0,
    "TOTAL_TRADES_3_MN": 0,
    "TOTAL_TRADES_6_MN": 0,
    "TOTAL_TRADES_12_MN": 0,

    "TOTAL_HL_TRADES": 0,
    "TOTAL_GL_TRADES": 0,
    "TOTAL_PL_TRADES": 0,
    "TOTAL_LIVE_PL_TRADES": 0,

    "TOTAL_PL_TRADES_1_MN": 0,
    "TOTAL_PL_TRADES_3_MN": 0,
    "TOTAL_PL_TRADES_6_MN": 0,
    "TOTAL_PL_TRADES_12_MN": 0,

    "TOTAL_CC_TRADES": 0,
    "TOTAL_CC_TRADES_1_MN": 0,
    "TOTAL_CC_TRADES_3_MN": 0,
    "TOTAL_CC_TRADES_6_MN": 0,
    "TOTAL_CC_TRADES_12_MN": 0,

    "TOTAL_SEC_TRADES": 0,
    "TOTAL_UNSEC_TRADES": 0,
    "TOTAL_UNSEC_TRADES_1_MN": 0,
    "TOTAL_UNSEC_TRADES_3_MN": 0,
    "TOTAL_UNSEC_TRADES_6_MN": 0,
    "TOTAL_UNSEC_TRADES_12_MN": 0,

    "TOTAL_INSTALLMENT_TRADES": 0,
    "TOTAL_LIVE_TRADES_M0_M2": 0,
    "TOTAL_LIVE_UNSEC_TRADES_M0_M2": 0,
    "TOTAL_LIVE_INSTALLMENT_TRADES_M0_M2": 0,

    # Balance / Amount (32-48)
    "TOTAL_BALANCE": 0,
    "TOTAL_PL_BALANCE": 0,
    "TOTAL_BL_BALANCE": 0,
    "TOTAL_CD_BALANCE": 0,
    "TOTAL_AL_BALANCE": 0,
    "TOTAL_HL_BALANCE": 0,
    "TOTAL_CC_BALANCE": 0,

    "TOTAL_SANC_AMT": 0,
    "TOTAL_PL_SANC_AMT": 0,
    "TOTAL_BL_SANC_AMT": 0,
    "TOTAL_CD_SANC_AMT": 0,
    "TOTAL_AL_SANC_AMT": 0,
    "TOTAL_HL_SANC_AMT": 0,
    "TOTAL_CC_SANC_AMT": 0,

    "TOTAL_AMT_PAST_DUE": 0,
    "TOTAL_WO_SF_TRADES": 0,
    "TOTAL_WO_SF_36O_TRADES": 0,

    # Utilization (49-50)
    "MACRO_UTIL": -1,
    "MICRO_UTIL": -1,

    # Days Since Open (51-70)
    "AVG_DAYS_SO_IN_LAST_6_MN": -1,
    "AVG_DAYS_SO_IN_LAST_12_MN": -1,
    "AVG_DAYS_SO_IN_LAST_36_MN": -1,

    "MEDIAN_DAYS_SO_IN_LAST_6_MN": -1,
    "MEDIAN_DAYS_SO_IN_LAST_12_MN": -1,
    "MEDIAN_DAYS_SO_IN_LAST_36_MN": -1,

    "AVG_DAYS_SO_UNSECURED_IN_LAST_6_MN": -1,
    "AVG_DAYS_SO_UNSECURED_IN_LAST_12_MN": -1,
    "AVG_DAYS_SO_UNSECURED_IN_LAST_36_MN": -1,

    "MEDIAN_DAYS_SO_UNSECURED_IN_LAST_6_MN": -1,
    "MEDIAN_DAYS_SO_UNSECURED_IN_LAST_12_MN": -1,
    "MEDIAN_DAYS_SO_UNSECURED_IN_LAST_36_MN": -1,

    "AVG_DAYS_SO_PL_IN_LAST_6_MN": -1,
    "AVG_DAYS_SO_PL_IN_LAST_12_MN": -1,
    "AVG_DAYS_SO_PL_IN_LAST_36_MN": -1,

    "MEDIAN_DAYS_SO_PL_IN_LAST_6_MN": -1,
    "MEDIAN_DAYS_SO_PL_IN_LAST_12_MN": -1,
    "MEDIAN_DAYS_SO_PL_IN_LAST_36_MN": -1,

    # Average Vintage (71-74)
    "AVG_PL_BUR_VINTAGE": -1,
    "AVG_BL_BUR_VINTAGE": -1,
    "AVG_CC_BUR_VINTAGE": -1,
    "AVG_CD_BUR_VINTAGE": -1,

    # H Features (75-116)
    "H0003": 0,
    "H0012": 0,
    "H0036": 0,
    "H0103": 0,
    "H0112": 0,
    "H0136": 0,
    "H0203": 0,
    "H0212": 0,
    "H0236": 0,
    "H0303": 0,
    "H0312": 0,
    "H0336": 0,
    "H0403": 0,
    "H0412": 0,
    "H0436": 0,
    "H0503": 0,
    "H0512": 0,
    "H0536": 0,
    "H1203": 0,
    "H1212": 0,
    "H1236": 0,
    "H1403": 0,
    "H1412": 0,
    "H1436": 0,
    "H1503": 0,
    "H1512": 0,
    "H1536": 0,
    "H2203": 0,
    "H2212": 0,
    "H2236": 0,
    "H2403": 0,
    "H2412": 0,
    "H2436": 0,
    "H2503": 0,
    "H2512": 0,
    "H2536": 0,
    "H3203": 0,
    "H3212": 0,
    "H3236": 0,
    "H3403": 0,
    "H3412": 0,
    "H3436": 0,
    "H3503": 0,
    "H3512": 0,
    "H3536": 0,

    # Max / Median DPD (117-125)
    "H4103": -1,
    "H4112": -1,
    "H4136": -1,
    "H4203": -1,
    "H4212": -1,
    "H4236": -1,
    "H4303": -1,
    "H4312": -1,
    "H4336": -1,

    # Trended DPD (126-131)
    "DPD_RECOVERY_RATE": -1,
    "DPD_SLOPE_6M": -1,
    "DPD_SLOPE_12M": -1,
    "DPD_CURRENT_VS_HISTORY_RATIO": -1,
    "NUM_CONSEC_MONTHS_ZERO_DPD": 0,

    # Trended Account (132-134)
    "ACCOUNT_CLOSURE_RATE_6M": -1,
    "PCT_NEW_UNSECURED_LOANS_6M": -1,
    "PCT_NEW_SECURED_LOANS_6M": -1
})

In [132]:
ref.select("user_id").count()

72314

In [133]:
ref.select("user_id").distinct().count()

72314

In [134]:
trade.select("user_id").distinct().count()

70507

In [135]:
enquiries.select("user_id").distinct().count()

71262

In [136]:
## case_study_users

In [137]:
case_study_users = spark.read.csv("case_study_submission_users_oot.csv", header=True, inferSchema=True)

In [138]:
case_study_users.count()

39962

In [139]:
case_study_users.distinct().count()

39962

In [140]:
case_study_users.show(2)

+-----------+
|    user_id|
+-----------+
|60129556744|
|34359738920|
+-----------+
only showing top 2 rows



In [141]:
trade_features.show(1, truncate=False)

26/06/30 18:21:54 WARN DAGScheduler: Broadcasting large task binary with size 1694.0 KiB
26/06/30 18:23:06 WARN DAGScheduler: Broadcasting large task binary with size 3.7 MiB


+-------+-------------+--------------+---------------+------------+-----------------+-----------------+-----------------+------------------+---------------+---------------+---------------+--------------------+--------------------+--------------------+--------------------+---------------------+---------------+--------------------+--------------------+--------------------+---------------------+----------------+------------------+-----------------------+-----------------------+-----------------------+------------------------+------------------------+-----------------------+-----------------------------+-----------------------------------+-------------+----------------+----------------+----------------+----------------+----------------+----------------+--------------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+------------------+------------------+----------------------+----------+----------+-----+-----+-----+-----+-----+-----

In [142]:
len(trade_features.columns)

135

In [143]:
########     #############     ################   ####################         ##########   #################   ####################    ####################
#### ##########   #############       ###########   ###################    ################    ###############      ##########     ##########################

In [144]:
trade_features.coalesce(1).write.mode("overwrite").parquet(
    "oot_trade_processed.parquet"
)

26/06/30 18:23:34 WARN DAGScheduler: Broadcasting large task binary with size 1695.0 KiB
26/06/30 18:24:41 WARN DAGScheduler: Broadcasting large task binary with size 3.9 MiB


In [145]:
########     #############     ################   ####################         ##########   #################   ####################    ####################
#### ##########   #############       ###########   ###################    ################    ###############      ##########     ##########################

## inquiry_features

In [146]:
enquiries.printSchema()

root
 |-- ACCOUNT_TYPE_CODE: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- DATE_OF_ENQUIRY: date (nullable = true)
 |-- ACCOUNT_TYPE: string (nullable = true)
 |-- ENQUIRY_AMOUNT: double (nullable = true)
 |-- retro_date: date (nullable = true)
 |-- BROAD_CATEGORY: string (nullable = false)
 |-- REVOLVING: integer (nullable = false)
 |-- SECURED: integer (nullable = false)



In [147]:
# Count null in each columns
enquiries.select([F.count(F.when(F.col(c).isNull(),c)).alias(c) for c in enquiries.columns]).show()

+-----------------+-------+---------------+------------+--------------+----------+--------------+---------+-------+
|ACCOUNT_TYPE_CODE|user_id|DATE_OF_ENQUIRY|ACCOUNT_TYPE|ENQUIRY_AMOUNT|retro_date|BROAD_CATEGORY|REVOLVING|SECURED|
+-----------------+-------+---------------+------------+--------------+----------+--------------+---------+-------+
|                0|      0|              0|           0|             0|         0|             0|        0|      0|
+-----------------+-------+---------------+------------+--------------+----------+--------------+---------+-------+



In [148]:
enquiries.agg(
    F.sum(F.when(F.col("DATE_OF_ENQUIRY") > F.col("retro_date"), 1).otherwise(0)).alias("DATE_OF_ENQUIRY > retro_date"),
    F.sum(F.when(F.col("DATE_OF_ENQUIRY") < F.col("retro_date"), 1).otherwise(0)).alias("retro_date > DATE_OF_ENQUIRY"),
    F.sum(F.when(F.col("DATE_OF_ENQUIRY") == F.col("retro_date"), 1).otherwise(0)).alias("DATE_OF_ENQUIRY = retro_date"),
    F.sum(F.when(F.col("DATE_OF_ENQUIRY").isNull() | F.col("retro_date").isNull(), 1).otherwise(0)).alias("null_dates")
).show(truncate=False)

+----------------------------+----------------------------+----------------------------+----------+
|DATE_OF_ENQUIRY > retro_date|retro_date > DATE_OF_ENQUIRY|DATE_OF_ENQUIRY = retro_date|null_dates|
+----------------------------+----------------------------+----------------------------+----------+
|0                           |1849472                     |0                           |0         |
+----------------------------+----------------------------+----------------------------+----------+



In [149]:
# Step 1: Compute previous enquiry date and gap (days) for each enquiry

w = Window.partitionBy("user_id").orderBy("DATE_OF_ENQUIRY")

enquiries_gap = enquiries.withColumn(
    "prev_date", F.lag("DATE_OF_ENQUIRY").over(w)
).withColumn(
    "prev_secured", F.lag("SECURED").over(w)
).withColumn(
    "prev_pl", F.lag(F.when(F.col("BROAD_CATEGORY") == "PL", 1).otherwise(0)).over(w)
).withColumn(
    "gap_days_all", F.datediff(F.col("DATE_OF_ENQUIRY"), F.col("prev_date"))
)

In [150]:
inquiry_features = enquiries_gap.groupBy("user_id").agg(
    F.max(F.months_between(F.col("retro_date"), F.col("DATE_OF_ENQUIRY"))).alias("G200"),
    F.sum(F.when(F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -1), 1).otherwise(0)).alias("G201"),
    F.sum(F.when(F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3), 1).otherwise(0)).alias("G203"),
    F.sum(F.when(F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12), 1).otherwise(0)).alias("G212"),
    F.sum(F.when(F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36), 1).otherwise(0)).alias("G236"),

    # 6 - 9
    F.sum(
        F.when(
            (F.col("SECURED") == 0) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -1)),
            1
        ).otherwise(0)
    ).alias("G301"),

    F.sum(
        F.when(
            (F.col("SECURED") == 0) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)),
            1
        ).otherwise(0)
    ).alias("G303"),

    F.sum(
        F.when(
            (F.col("SECURED") == 0) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)),
            1
        ).otherwise(0)
    ).alias("G312"),

    F.sum(
        F.when(
            (F.col("SECURED") == 0) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)),
            1
        ).otherwise(0)
    ).alias("G336"),

    # 10 - G401
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -1)),
            1
        ).otherwise(0)
    ).alias("G401"),

    # 11 - G403
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)),
            1
        ).otherwise(0)
    ).alias("G403"),

    # 12 - G412
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)),
            1
        ).otherwise(0)
    ).alias("G412"),

    # 13 - G436
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)),
            1
        ).otherwise(0)
    ).alias("G436"),

    # 14 - G5101
    F.sum(
        F.when(
            (F.col("ENQUIRY_AMOUNT") >= 10000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -1)),
            1
        ).otherwise(0)
    ).alias("G5101"),

    # 15 - G5103
    F.sum(
        F.when(
            (F.col("ENQUIRY_AMOUNT") >= 10000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)),
            1
        ).otherwise(0)
    ).alias("G5103"),

    # 16 - G5112
    F.sum(
        F.when(
            (F.col("ENQUIRY_AMOUNT") >= 10000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)),
            1
        ).otherwise(0)
    ).alias("G5112"),

    # 17 - G5136
    F.sum(
        F.when(
            (F.col("ENQUIRY_AMOUNT") >= 10000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)),
            1
        ).otherwise(0)
    ).alias("G5136"),

    # 18 - G5201
    F.sum(
        F.when(
            (F.col("ENQUIRY_AMOUNT") >= 50000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -1)),
            1
        ).otherwise(0)
    ).alias("G5201"),

    # 19 - G5203
    F.sum(
        F.when(
            (F.col("ENQUIRY_AMOUNT") >= 50000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)),
            1
        ).otherwise(0)
    ).alias("G5203"),

    # 20 - G5212
    F.sum(
        F.when(
            (F.col("ENQUIRY_AMOUNT") >= 50000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)),
            1
        ).otherwise(0)
    ).alias("G5212"),

    # 21 - G5236
    F.sum(
        F.when(
            (F.col("ENQUIRY_AMOUNT") >= 50000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)),
            1
        ).otherwise(0)
    ).alias("G5236"),

    # 22 - G5301
    F.sum(
        F.when(
            (F.col("ENQUIRY_AMOUNT") >= 100000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -1)),
            1
        ).otherwise(0)
    ).alias("G5301"),

    # 23 - G5303
    F.sum(
        F.when(
            (F.col("ENQUIRY_AMOUNT") >= 100000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)),
            1
        ).otherwise(0)
    ).alias("G5303"),

    # 24 - G5312
    F.sum(
        F.when(
            (F.col("ENQUIRY_AMOUNT") >= 100000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)),
            1
        ).otherwise(0)
    ).alias("G5312"),

    # 25 - G5336
    F.sum(
        F.when(
            (F.col("ENQUIRY_AMOUNT") >= 100000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)),
            1
        ).otherwise(0)
    ).alias("G5336"),

    # 26 - G5401
    F.sum(
        F.when(
            (F.col("ENQUIRY_AMOUNT") >= 500000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -1)),
            1
        ).otherwise(0)
    ).alias("G5401"),

    # 27 - G5403
    F.sum(
        F.when(
            (F.col("ENQUIRY_AMOUNT") >= 500000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)),
            1
        ).otherwise(0)
    ).alias("G5403"),

    # 28 - G5412
    F.sum(
        F.when(
            (F.col("ENQUIRY_AMOUNT") >= 500000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)),
            1
        ).otherwise(0)
    ).alias("G5412"),

    # 29 - G5436
    F.sum(
        F.when(
            (F.col("ENQUIRY_AMOUNT") >= 500000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)),
            1
        ).otherwise(0)
    ).alias("G5436"),

    # 30 - G6101
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("ENQUIRY_AMOUNT") >= 10000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -1)),
            1
        ).otherwise(0)
    ).alias("G6101"),

    # 31 - G6103
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("ENQUIRY_AMOUNT") >= 10000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)),
            1
        ).otherwise(0)
    ).alias("G6103"),

    # 32 - G6112
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("ENQUIRY_AMOUNT") >= 10000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)),
            1
        ).otherwise(0)
    ).alias("G6112"),

    # 33 - G6136
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("ENQUIRY_AMOUNT") >= 10000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)),
            1
        ).otherwise(0)
    ).alias("G6136"),

    # 34 - G6201
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("ENQUIRY_AMOUNT") >= 50000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -1)),
            1
        ).otherwise(0)
    ).alias("G6201"),

    # 35 - G6203
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("ENQUIRY_AMOUNT") >= 50000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)),
            1
        ).otherwise(0)
    ).alias("G6203"),

    # 36 - G6212
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("ENQUIRY_AMOUNT") >= 50000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)),
            1
        ).otherwise(0)
    ).alias("G6212"),

    # 37 - G6236
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("ENQUIRY_AMOUNT") >= 50000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)),
            1
        ).otherwise(0)
    ).alias("G6236"),

    # 38 - G6301
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("ENQUIRY_AMOUNT") >= 100000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -1)),
            1
        ).otherwise(0)
    ).alias("G6301"),

    # 39 - G6303
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("ENQUIRY_AMOUNT") >= 100000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)),
            1
        ).otherwise(0)
    ).alias("G6303"),

    # 40 - G6312
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("ENQUIRY_AMOUNT") >= 100000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)),
            1
        ).otherwise(0)
    ).alias("G6312"),

    # 41 - G6336
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("ENQUIRY_AMOUNT") >= 100000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)),
            1
        ).otherwise(0)
    ).alias("G6336"),

    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("ENQUIRY_AMOUNT") >= 500000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -1)),
            1
        ).otherwise(0)
    ).alias("G6401"),

    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("ENQUIRY_AMOUNT") >= 500000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)),
            1
        ).otherwise(0)
    ).alias("G6403"),

    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("ENQUIRY_AMOUNT") >= 500000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)),
            1
        ).otherwise(0)
    ).alias("G6412"),

    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("ENQUIRY_AMOUNT") >= 500000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)),
            1
        ).otherwise(0)
    ).alias("G6436"),

    # 46 - G7101
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "CD") &
            (F.col("ENQUIRY_AMOUNT") >= 10000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -1)),
            1
        ).otherwise(0)
    ).alias("G7101"),

    # 47 - G7103
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "CD") &
            (F.col("ENQUIRY_AMOUNT") >= 10000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)),
            1
        ).otherwise(0)
    ).alias("G7103"),

    # 48 - G7112
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "CD") &
            (F.col("ENQUIRY_AMOUNT") >= 10000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)),
            1
        ).otherwise(0)
    ).alias("G7112"),

    # 49 - G7136
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "CD") &
            (F.col("ENQUIRY_AMOUNT") >= 10000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)),
            1
        ).otherwise(0)
    ).alias("G7136"),

    # 50 - G7201
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "CD") &
            (F.col("ENQUIRY_AMOUNT") >= 50000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -1)),
            1
        ).otherwise(0)
    ).alias("G7201"),

    # 51 - G7203
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "CD") &
            (F.col("ENQUIRY_AMOUNT") >= 50000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)),
            1
        ).otherwise(0)
    ).alias("G7203"),

    # 52 - G7212
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "CD") &
            (F.col("ENQUIRY_AMOUNT") >= 50000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)),
            1
        ).otherwise(0)
    ).alias("G7212"),

    # 53 - G7236
    F.sum(
        F.when(
            (F.col("BROAD_CATEGORY") == "CD") &
            (F.col("ENQUIRY_AMOUNT") >= 50000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)),
            1
        ).otherwise(0)
    ).alias("G7236"),

    # 54 - G8101
    F.sum(
        F.when(
            (F.col("SECURED") == 0) &
            (F.col("ENQUIRY_AMOUNT") >= 10000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -1)),
            1
        ).otherwise(0)
    ).alias("G8101"),
    
    # 55 - G8103
    F.sum(
        F.when(
            (F.col("SECURED") == 0) &
            (F.col("ENQUIRY_AMOUNT") >= 10000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)),
            1
        ).otherwise(0)
    ).alias("G8103"),
    
    # 56 - G8112
    F.sum(
        F.when(
            (F.col("SECURED") == 0) &
            (F.col("ENQUIRY_AMOUNT") >= 10000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)),
            1
        ).otherwise(0)
    ).alias("G8112"),
    
    # 57 - G8136
    F.sum(
        F.when(
            (F.col("SECURED") == 0) &
            (F.col("ENQUIRY_AMOUNT") >= 10000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)),
            1
        ).otherwise(0)
    ).alias("G8136"),
    
    # 58 - G8201
    F.sum(
        F.when(
            (F.col("SECURED") == 0) &
            (F.col("ENQUIRY_AMOUNT") >= 50000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -1)),
            1
        ).otherwise(0)
    ).alias("G8201"),
    
    # 59 - G8203
    F.sum(
        F.when(
            (F.col("SECURED") == 0) &
            (F.col("ENQUIRY_AMOUNT") >= 50000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)),
            1
        ).otherwise(0)
    ).alias("G8203"),
    
    # 60 - G8212
    F.sum(
        F.when(
            (F.col("SECURED") == 0) &
            (F.col("ENQUIRY_AMOUNT") >= 50000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)),
            1
        ).otherwise(0)
    ).alias("G8212"),
    
    # 61 - G8236
    F.sum(
        F.when(
            (F.col("SECURED") == 0) &
            (F.col("ENQUIRY_AMOUNT") >= 50000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)),
            1
        ).otherwise(0)
    ).alias("G8236"),
    
    # 62 - G8301
    F.sum(
        F.when(
            (F.col("SECURED") == 0) &
            (F.col("ENQUIRY_AMOUNT") >= 100000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -1)),
            1
        ).otherwise(0)
    ).alias("G8301"),
    
    # 63 - G8303
    F.sum(
        F.when(
            (F.col("SECURED") == 0) &
            (F.col("ENQUIRY_AMOUNT") >= 100000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)),
            1
        ).otherwise(0)
    ).alias("G8303"),
    
    # 64 - G8312
    F.sum(
        F.when(
            (F.col("SECURED") == 0) &
            (F.col("ENQUIRY_AMOUNT") >= 100000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)),
            1
        ).otherwise(0)
    ).alias("G8312"),
    
    # 65 - G8336
    F.sum(
        F.when(
            (F.col("SECURED") == 0) &
            (F.col("ENQUIRY_AMOUNT") >= 100000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)),
            1
        ).otherwise(0)
    ).alias("G8336"),
    
    # 66 - G8401
    F.sum(
        F.when(
            (F.col("SECURED") == 0) &
            (F.col("ENQUIRY_AMOUNT") >= 500000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -1)),
            1
        ).otherwise(0)
    ).alias("G8401"),
    
    # 67 - G8403
    F.sum(
        F.when(
            (F.col("SECURED") == 0) &
            (F.col("ENQUIRY_AMOUNT") >= 500000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)),
            1
        ).otherwise(0)
    ).alias("G8403"),
    
    # 68 - G8412
    F.sum(
        F.when(
            (F.col("SECURED") == 0) &
            (F.col("ENQUIRY_AMOUNT") >= 500000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)),
            1
        ).otherwise(0)
    ).alias("G8412"),
    
    # 69 - G8436
    F.sum(
        F.when(
            (F.col("SECURED") == 0) &
            (F.col("ENQUIRY_AMOUNT") >= 500000) &
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)),
            1
        ).otherwise(0)
    ).alias("G8436"),


     # GAP STATISTICS (G9103, G9112, G9136 – avg; G9203, G9212, G9236 – max; G9303, G9312, G9336 – min)
    # We include only gaps where both the current and previous enquiry fall within the window.
    # This matches the separate-filter approach.
    
    # 3-month window
    F.avg(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -3)) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G9103"),
    F.max(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -3)) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G9203"),
    F.min(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -3)) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G9303"),
    
    # 12-month window
    F.avg(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -12)) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G9112"),
    F.max(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -12)) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G9212"),
    F.min(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -12)) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G9312"),
    
    # 36-month window
    F.avg(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -36)) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G9136"),
    F.max(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -36)) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G9236"),
    F.min(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -36)) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G9336"),


    # UNECURED GAP STATISTICS (G10103, G10112, G10136 – avg; G10203, G10212, G10236 – max; G10303, G10312, G10336 – min)
    # 3-month window
    F.avg(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -3)) &
            (F.col("SECURED") == 0) &
            (F.col("prev_secured") == 0) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G10103"),
    F.max(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -3)) &
            (F.col("SECURED") == 0) &
            (F.col("prev_secured") == 0) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G10203"),
    F.min(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -3)) &
            (F.col("SECURED") == 0) &
            (F.col("prev_secured") == 0) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G10303"),
    
    # 12-month window
    F.avg(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -12)) &
            (F.col("SECURED") == 0) &
            (F.col("prev_secured") == 0) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G10112"),
    F.max(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -12)) &
            (F.col("SECURED") == 0) &
            (F.col("prev_secured") == 0) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G10212"),
    F.min(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -12)) &
            (F.col("SECURED") == 0) &
            (F.col("prev_secured") == 0) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G10312"),
    
    # 36-month window
    F.avg(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -36)) &
            (F.col("SECURED") == 0) &
            (F.col("prev_secured") == 0) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G10136"),
    F.max(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -36)) &
            (F.col("SECURED") == 0) &
            (F.col("prev_secured") == 0) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G10236"),
    F.min(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -36)) &
            (F.col("SECURED") == 0) &
            (F.col("prev_secured") == 0) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G10336"),



    # PL GAP STATISTICS (G11103, G11112, G11136 – avg; G11203, G11212, G11236 – max; G11303, G11312, G11336 – min)
    # ===============================================================
    # 3-month window
    F.avg(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -3)) &
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("prev_pl") == 1) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G11103"),
    F.max(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -3)) &
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("prev_pl") == 1) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G11203"),
    F.min(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -3)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -3)) &
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("prev_pl") == 1) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G11303"),
    
    # 12-month window
    F.avg(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -12)) &
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("prev_pl") == 1) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G11112"),
    F.max(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -12)) &
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("prev_pl") == 1) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G11212"),
    F.min(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -12)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -12)) &
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("prev_pl") == 1) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G11312"),
    
    # 36-month window
    F.avg(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -36)) &
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("prev_pl") == 1) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G11136"),
    F.max(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -36)) &
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("prev_pl") == 1) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G11236"),
    F.min(
        F.when(
            (F.col("DATE_OF_ENQUIRY") >= F.add_months(F.col("retro_date"), -36)) &
            (F.col("prev_date") >= F.add_months(F.col("retro_date"), -36)) &
            (F.col("BROAD_CATEGORY") == "PL") &
            (F.col("prev_pl") == 1) &
            F.col("gap_days_all").isNotNull(),
            F.col("gap_days_all")
        )
    ).alias("G11336")
    
)

In [151]:
# 97 MONTH_ON_MONTH_INCREASE_IN_ENQUIRIES_12M

In [152]:
monthly_counts = enquiries.groupBy("user_id").agg(
    *[
        F.sum(
            F.when(
                (
                    (F.year("retro_date") - F.year("DATE_OF_ENQUIRY")) * 12 +
                    (F.month("retro_date") - F.month("DATE_OF_ENQUIRY"))
                ) == i,
                1
            ).otherwise(0)
        ).alias(f"cnt_m{i}")
        for i in range(12)
    ]
)

monthly_increase = monthly_counts.select(
    "user_id",
    (
        F.when(F.col("cnt_m0") > F.col("cnt_m1"), 1).otherwise(0) +
        F.when(F.col("cnt_m1") > F.col("cnt_m2"), 1).otherwise(0) +
        F.when(F.col("cnt_m2") > F.col("cnt_m3"), 1).otherwise(0) +
        F.when(F.col("cnt_m3") > F.col("cnt_m4"), 1).otherwise(0) +
        F.when(F.col("cnt_m4") > F.col("cnt_m5"), 1).otherwise(0) +
        F.when(F.col("cnt_m5") > F.col("cnt_m6"), 1).otherwise(0) +
        F.when(F.col("cnt_m6") > F.col("cnt_m7"), 1).otherwise(0) +
        F.when(F.col("cnt_m7") > F.col("cnt_m8"), 1).otherwise(0) +
        F.when(F.col("cnt_m8") > F.col("cnt_m9"), 1).otherwise(0) +
        F.when(F.col("cnt_m9") > F.col("cnt_m10"), 1).otherwise(0) +
        F.when(F.col("cnt_m10") > F.col("cnt_m11"), 1).otherwise(0)
    ).alias("MONTH_ON_MONTH_INCREASE_IN_ENQUIRIES_12M")
)

In [153]:
inquiry_features = inquiry_features.join(monthly_increase, on="user_id", how="left")

In [154]:
# 98 RECENT_ENQUIRY_SPIKE_3M

In [155]:
recent_spike_3m = monthly_counts.select(
    "user_id",
    F.when(
        (F.col("cnt_m3") + F.col("cnt_m4") + F.col("cnt_m5")) == 0,
        F.lit(-1)
    ).otherwise(
        (
            (F.col("cnt_m0") + F.col("cnt_m1") + F.col("cnt_m2")) -
            (F.col("cnt_m3") + F.col("cnt_m4") + F.col("cnt_m5"))
        ) /
        (F.col("cnt_m3") + F.col("cnt_m4") + F.col("cnt_m5"))
    ).alias("RECENT_ENQUIRY_SPIKE_3M")
)

In [156]:
inquiry_features = inquiry_features.join(recent_spike_3m, on="user_id", how="left")

In [157]:
# CONSECUTIVE_HIGH_ENQUIRY_MONTHS_12M

In [158]:
consecutive_high = monthly_counts.select(
    "user_id",
    F.expr("""
        aggregate(
            array(
                IF(cnt_m0  > 1, 1, 0),
                IF(cnt_m1  > 1, 1, 0),
                IF(cnt_m2  > 1, 1, 0),
                IF(cnt_m3  > 1, 1, 0),
                IF(cnt_m4  > 1, 1, 0),
                IF(cnt_m5  > 1, 1, 0),
                IF(cnt_m6  > 1, 1, 0),
                IF(cnt_m7  > 1, 1, 0),
                IF(cnt_m8  > 1, 1, 0),
                IF(cnt_m9  > 1, 1, 0),
                IF(cnt_m10 > 1, 1, 0),
                IF(cnt_m11 > 1, 1, 0)
            ),
            named_struct('cur', 0, 'best', 0),
            (acc, x) ->
                named_struct(
                    'cur',  IF(x = 1, acc.cur + 1, 0),
                    'best', greatest(acc.best, IF(x = 1, acc.cur + 1, 0))
                ),
            acc -> acc.best
        )
    """).alias("CONSECUTIVE_HIGH_ENQUIRY_MONTHS_12M")
)

In [159]:
inquiry_features = inquiry_features.join(consecutive_high, on="user_id", how="left")

In [160]:
# Filling Missing Values

inquiry_features = inquiry_features.fillna({

    # 1-5
    "G200": 99999,
    "G201": 0,
    "G203": 0,
    "G212": 0,
    "G236": 0,

    # 6-13
    "G301": 0,
    "G303": 0,
    "G312": 0,
    "G336": 0,
    "G401": 0,
    "G403": 0,
    "G412": 0,
    "G436": 0,

    # 14-29
    "G5101": 0,
    "G5103": 0,
    "G5112": 0,
    "G5136": 0,
    "G5201": 0,
    "G5203": 0,
    "G5212": 0,
    "G5236": 0,
    "G5301": 0,
    "G5303": 0,
    "G5312": 0,
    "G5336": 0,
    "G5401": 0,
    "G5403": 0,
    "G5412": 0,
    "G5436": 0,

    # 30-45
    "G6101": 0,
    "G6103": 0,
    "G6112": 0,
    "G6136": 0,
    "G6201": 0,
    "G6203": 0,
    "G6212": 0,
    "G6236": 0,
    "G6301": 0,
    "G6303": 0,
    "G6312": 0,
    "G6336": 0,
    "G6401": 0,
    "G6403": 0,
    "G6412": 0,
    "G6436": 0,

    # 46-53
    "G7101": 0,
    "G7103": 0,
    "G7112": 0,
    "G7136": 0,
    "G7201": 0,
    "G7203": 0,
    "G7212": 0,
    "G7236": 0,

    # 54-69
    "G8101": 0,
    "G8103": 0,
    "G8112": 0,
    "G8136": 0,
    "G8201": 0,
    "G8203": 0,
    "G8212": 0,
    "G8236": 0,
    "G8301": 0,
    "G8303": 0,
    "G8312": 0,
    "G8336": 0,
    "G8401": 0,
    "G8403": 0,
    "G8412": 0,
    "G8436": 0,

    # 70-96
    "G9103": -1,
    "G9112": -1,
    "G9136": -1,

    "G9203": -1,
    "G9212": -1,
    "G9236": -1,

    "G9303": -1,
    "G9312": -1,
    "G9336": -1,

    "G10103": -1,
    "G10112": -1,
    "G10136": -1,

    "G10203": -1,
    "G10212": -1,
    "G10236": -1,

    "G10303": -1,
    "G10312": -1,
    "G10336": -1,

    "G11103": -1,
    "G11112": -1,
    "G11136": -1,

    "G11203": -1,
    "G11212": -1,
    "G11236": -1,

    "G11303": -1,
    "G11312": -1,
    "G11336": -1,

    # 97-99
    "MONTH_ON_MONTH_INCREASE_IN_ENQUIRIES_12M": 0,
    "RECENT_ENQUIRY_SPIKE_3M": -1,
    "CONSECUTIVE_HIGH_ENQUIRY_MONTHS_12M": 0

})

In [161]:
len(inquiry_features.columns)

100

In [162]:
########     #############     ################   ####################         ##########   #################   ####################    ####################
#### ##########   #############       ###########   ###################    ################    ###############      ##########     ##########################

In [163]:
inquiry_features.coalesce(1).write.mode("overwrite").parquet(
    "oot_inq_processed.parquet"
)

In [164]:
########     #############     ################   ####################         ##########   #################   ####################    ####################
#### ##########   #############       ###########   ###################    ################    ###############      ##########     ##########################

In [165]:
enquiries.printSchema()

root
 |-- ACCOUNT_TYPE_CODE: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- DATE_OF_ENQUIRY: date (nullable = true)
 |-- ACCOUNT_TYPE: string (nullable = true)
 |-- ENQUIRY_AMOUNT: double (nullable = true)
 |-- retro_date: date (nullable = true)
 |-- BROAD_CATEGORY: string (nullable = false)
 |-- REVOLVING: integer (nullable = false)
 |-- SECURED: integer (nullable = false)



In [166]:
## ENQUIRY_TO_OPEN_ACCOUNT_RATIO

In [167]:
# Step 1: All enquiry count before retro_date
enq_cnt = enquiries.groupBy("user_id").agg(
    F.sum(
        F.when(F.col("DATE_OF_ENQUIRY") < F.col("retro_date"), 1).otherwise(0)
    ).alias("ENQ_CNT")
)

In [168]:
# Step 2: All open account count before retro_date
open_cnt = trade.groupBy("user_id").agg(
    F.sum(
        F.when(F.col("OPEN_DATE") < F.col("retro_date"), 1).otherwise(0)
    ).alias("OPEN_CNT")
)

In [169]:
# Step 3: Join and compute ratio
enquiry_to_open_ratio = (
    case_study_users
    .join(open_cnt, on="user_id", how="left")
    .join(enq_cnt, on="user_id", how="left")
    .fillna({"ENQ_CNT": 0, "OPEN_CNT": 0})
    .select(
        "user_id",
        F.when(F.col("OPEN_CNT") > 0,
               F.col("ENQ_CNT") / F.col("OPEN_CNT"))
         .otherwise(-1).alias("ENQUIRY_TO_OPEN_ACCOUNT_RATIO")
    )
)

In [170]:
final_features = (
    case_study_users
    .join(trade_features, on="user_id", how="left")
    .join(inquiry_features, on="user_id", how="left")
    .join(enquiry_to_open_ratio.select("user_id", "ENQUIRY_TO_OPEN_ACCOUNT_RATIO"), on="user_id", how="left")
    .fillna({"ENQUIRY_TO_OPEN_ACCOUNT_RATIO": -1})
)

In [171]:
len(final_features.columns)

235

In [172]:
########     #############     ################   ####################         ##########   #################   ####################    ####################
#### ##########   #############       ###########   ###################    ################    ###############      ##########     ##########################

In [173]:
# Fill default Values

final_features = final_features.fillna({

    # Vintage (1-3)
    "FIRST_PRODUCT": "Other",
    "LATEST_PRODUCT": "Other",
    "MAX_BUR_VINTAGE": -1,

    # Counts (4-31)
    "TOTAL_TRADES": 0,
    "TOTAL_TRADES_1_MN": 0,
    "TOTAL_TRADES_3_MN": 0,
    "TOTAL_TRADES_6_MN": 0,
    "TOTAL_TRADES_12_MN": 0,

    "TOTAL_HL_TRADES": 0,
    "TOTAL_GL_TRADES": 0,
    "TOTAL_PL_TRADES": 0,
    "TOTAL_LIVE_PL_TRADES": 0,

    "TOTAL_PL_TRADES_1_MN": 0,
    "TOTAL_PL_TRADES_3_MN": 0,
    "TOTAL_PL_TRADES_6_MN": 0,
    "TOTAL_PL_TRADES_12_MN": 0,

    "TOTAL_CC_TRADES": 0,
    "TOTAL_CC_TRADES_1_MN": 0,
    "TOTAL_CC_TRADES_3_MN": 0,
    "TOTAL_CC_TRADES_6_MN": 0,
    "TOTAL_CC_TRADES_12_MN": 0,

    "TOTAL_SEC_TRADES": 0,
    "TOTAL_UNSEC_TRADES": 0,
    "TOTAL_UNSEC_TRADES_1_MN": 0,
    "TOTAL_UNSEC_TRADES_3_MN": 0,
    "TOTAL_UNSEC_TRADES_6_MN": 0,
    "TOTAL_UNSEC_TRADES_12_MN": 0,

    "TOTAL_INSTALLMENT_TRADES": 0,
    "TOTAL_LIVE_TRADES_M0_M2": 0,
    "TOTAL_LIVE_UNSEC_TRADES_M0_M2": 0,
    "TOTAL_LIVE_INSTALLMENT_TRADES_M0_M2": 0,

    # Balance / Amount (32-48)
    "TOTAL_BALANCE": 0,
    "TOTAL_PL_BALANCE": 0,
    "TOTAL_BL_BALANCE": 0,
    "TOTAL_CD_BALANCE": 0,
    "TOTAL_AL_BALANCE": 0,
    "TOTAL_HL_BALANCE": 0,
    "TOTAL_CC_BALANCE": 0,

    "TOTAL_SANC_AMT": 0,
    "TOTAL_PL_SANC_AMT": 0,
    "TOTAL_BL_SANC_AMT": 0,
    "TOTAL_CD_SANC_AMT": 0,
    "TOTAL_AL_SANC_AMT": 0,
    "TOTAL_HL_SANC_AMT": 0,
    "TOTAL_CC_SANC_AMT": 0,

    "TOTAL_AMT_PAST_DUE": 0,
    "TOTAL_WO_SF_TRADES": 0,
    "TOTAL_WO_SF_36O_TRADES": 0,

    # Utilization (49-50)
    "MACRO_UTIL": -1,
    "MICRO_UTIL": -1,

    # Days Since Open (51-70)
    "AVG_DAYS_SO_IN_LAST_6_MN": -1,
    "AVG_DAYS_SO_IN_LAST_12_MN": -1,
    "AVG_DAYS_SO_IN_LAST_36_MN": -1,

    "MEDIAN_DAYS_SO_IN_LAST_6_MN": -1,
    "MEDIAN_DAYS_SO_IN_LAST_12_MN": -1,
    "MEDIAN_DAYS_SO_IN_LAST_36_MN": -1,

    "AVG_DAYS_SO_UNSECURED_IN_LAST_6_MN": -1,
    "AVG_DAYS_SO_UNSECURED_IN_LAST_12_MN": -1,
    "AVG_DAYS_SO_UNSECURED_IN_LAST_36_MN": -1,

    "MEDIAN_DAYS_SO_UNSECURED_IN_LAST_6_MN": -1,
    "MEDIAN_DAYS_SO_UNSECURED_IN_LAST_12_MN": -1,
    "MEDIAN_DAYS_SO_UNSECURED_IN_LAST_36_MN": -1,

    "AVG_DAYS_SO_PL_IN_LAST_6_MN": -1,
    "AVG_DAYS_SO_PL_IN_LAST_12_MN": -1,
    "AVG_DAYS_SO_PL_IN_LAST_36_MN": -1,

    "MEDIAN_DAYS_SO_PL_IN_LAST_6_MN": -1,
    "MEDIAN_DAYS_SO_PL_IN_LAST_12_MN": -1,
    "MEDIAN_DAYS_SO_PL_IN_LAST_36_MN": -1,

    # Average Vintage (71-74)
    "AVG_PL_BUR_VINTAGE": -1,
    "AVG_BL_BUR_VINTAGE": -1,
    "AVG_CC_BUR_VINTAGE": -1,
    "AVG_CD_BUR_VINTAGE": -1,

    # H Features (75-116)
    "H0003": 0,
    "H0012": 0,
    "H0036": 0,
    "H0103": 0,
    "H0112": 0,
    "H0136": 0,
    "H0203": 0,
    "H0212": 0,
    "H0236": 0,
    "H0303": 0,
    "H0312": 0,
    "H0336": 0,
    "H0403": 0,
    "H0412": 0,
    "H0436": 0,
    "H0503": 0,
    "H0512": 0,
    "H0536": 0,
    "H1203": 0,
    "H1212": 0,
    "H1236": 0,
    "H1403": 0,
    "H1412": 0,
    "H1436": 0,
    "H1503": 0,
    "H1512": 0,
    "H1536": 0,
    "H2203": 0,
    "H2212": 0,
    "H2236": 0,
    "H2403": 0,
    "H2412": 0,
    "H2436": 0,
    "H2503": 0,
    "H2512": 0,
    "H2536": 0,
    "H3203": 0,
    "H3212": 0,
    "H3236": 0,
    "H3403": 0,
    "H3412": 0,
    "H3436": 0,
    "H3503": 0,
    "H3512": 0,
    "H3536": 0,

    # Max / Median DPD (117-125)
    "H4103": -1,
    "H4112": -1,
    "H4136": -1,
    "H4203": -1,
    "H4212": -1,
    "H4236": -1,
    "H4303": -1,
    "H4312": -1,
    "H4336": -1,

    # Trended DPD (126-131)
    "DPD_RECOVERY_RATE": -1,
    "DPD_SLOPE_6M": -1,
    "DPD_SLOPE_12M": -1,
    "DPD_CURRENT_VS_HISTORY_RATIO": -1,
    "NUM_CONSEC_MONTHS_ZERO_DPD": 0,

    # Trended Account (132-134)
    "ACCOUNT_CLOSURE_RATE_6M": -1,
    "PCT_NEW_UNSECURED_LOANS_6M": -1,
    "PCT_NEW_SECURED_LOANS_6M": -1
})

In [174]:
# Filling Missing Values

features = final_features.fillna({

    # 1-5
    "G200": 99999,
    "G201": 0,
    "G203": 0,
    "G212": 0,
    "G236": 0,

    # 6-13
    "G301": 0,
    "G303": 0,
    "G312": 0,
    "G336": 0,
    "G401": 0,
    "G403": 0,
    "G412": 0,
    "G436": 0,

    # 14-29
    "G5101": 0,
    "G5103": 0,
    "G5112": 0,
    "G5136": 0,
    "G5201": 0,
    "G5203": 0,
    "G5212": 0,
    "G5236": 0,
    "G5301": 0,
    "G5303": 0,
    "G5312": 0,
    "G5336": 0,
    "G5401": 0,
    "G5403": 0,
    "G5412": 0,
    "G5436": 0,

    # 30-45
    "G6101": 0,
    "G6103": 0,
    "G6112": 0,
    "G6136": 0,
    "G6201": 0,
    "G6203": 0,
    "G6212": 0,
    "G6236": 0,
    "G6301": 0,
    "G6303": 0,
    "G6312": 0,
    "G6336": 0,
    "G6401": 0,
    "G6403": 0,
    "G6412": 0,
    "G6436": 0,

    # 46-53
    "G7101": 0,
    "G7103": 0,
    "G7112": 0,
    "G7136": 0,
    "G7201": 0,
    "G7203": 0,
    "G7212": 0,
    "G7236": 0,

    # 54-69
    "G8101": 0,
    "G8103": 0,
    "G8112": 0,
    "G8136": 0,
    "G8201": 0,
    "G8203": 0,
    "G8212": 0,
    "G8236": 0,
    "G8301": 0,
    "G8303": 0,
    "G8312": 0,
    "G8336": 0,
    "G8401": 0,
    "G8403": 0,
    "G8412": 0,
    "G8436": 0,

    # 70-96
    "G9103": -1,
    "G9112": -1,
    "G9136": -1,

    "G9203": -1,
    "G9212": -1,
    "G9236": -1,

    "G9303": -1,
    "G9312": -1,
    "G9336": -1,

    "G10103": -1,
    "G10112": -1,
    "G10136": -1,

    "G10203": -1,
    "G10212": -1,
    "G10236": -1,

    "G10303": -1,
    "G10312": -1,
    "G10336": -1,

    "G11103": -1,
    "G11112": -1,
    "G11136": -1,

    "G11203": -1,
    "G11212": -1,
    "G11236": -1,

    "G11303": -1,
    "G11312": -1,
    "G11336": -1,

    # 97-99
    "MONTH_ON_MONTH_INCREASE_IN_ENQUIRIES_12M": 0,
    "RECENT_ENQUIRY_SPIKE_3M": -1,
    "CONSECUTIVE_HIGH_ENQUIRY_MONTHS_12M": 0

})

In [175]:
final_features = final_features.withColumnRenamed("user_id", "id")

In [176]:
final_features.coalesce(1).write.mode("overwrite").parquet(
    "oot_features.parquet"
)

26/06/30 18:27:34 WARN DAGScheduler: Broadcasting large task binary with size 1695.0 KiB
26/06/30 18:29:08 WARN DAGScheduler: Broadcasting large task binary with size 4.3 MiB


In [177]:
########     #############     ################   ####################         ##########   #################   ####################    ####################
#### ##########   #############       ###########   ###################    ################    ###############      ##########     ##########################

In [178]:
len(final_features.columns)

235

26/06/30 18:35:31 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/blockmgr-3902c416-64ca-4aac-9d26-3c906197edbe. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/blockmgr-3902c416-64ca-4aac-9d26-3c906197edbe
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:173)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:109)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:90)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1126)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1(DiskBlockManager.scala:368)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1$adapted(DiskBlockManager.scala:364)
	at scala.collection.IndexedSeqOptimize

In [173]:
# final_features = spark.read.parquet("final_features.parquet")

In [174]:
# final_features.columns

['user_id',
 'FIRST_PRODUCT',
 'LATEST_PRODUCT',
 'MAX_BUR_VINTAGE',
 'TOTAL_TRADES',
 'TOTAL_TRADES_1_MN',
 'TOTAL_TRADES_3_MN',
 'TOTAL_TRADES_6_MN',
 'TOTAL_TRADES_12_MN',
 'TOTAL_HL_TRADES',
 'TOTAL_GL_TRADES',
 'TOTAL_PL_TRADES',
 'TOTAL_LIVE_PL_TRADES',
 'TOTAL_PL_TRADES_1_MN',
 'TOTAL_PL_TRADES_3_MN',
 'TOTAL_PL_TRADES_6_MN',
 'TOTAL_PL_TRADES_12_MN',
 'TOTAL_CC_TRADES',
 'TOTAL_CC_TRADES_1_MN',
 'TOTAL_CC_TRADES_3_MN',
 'TOTAL_CC_TRADES_6_MN',
 'TOTAL_CC_TRADES_12_MN',
 'TOTAL_SEC_TRADES',
 'TOTAL_UNSEC_TRADES',
 'TOTAL_UNSEC_TRADES_1_MN',
 'TOTAL_UNSEC_TRADES_3_MN',
 'TOTAL_UNSEC_TRADES_6_MN',
 'TOTAL_UNSEC_TRADES_12_MN',
 'TOTAL_INSTALLMENT_TRADES',
 'TOTAL_LIVE_TRADES_M0_M2',
 'TOTAL_LIVE_UNSEC_TRADES_M0_M2',
 'TOTAL_LIVE_INSTALLMENT_TRADES_M0_M2',
 'TOTAL_BALANCE',
 'TOTAL_PL_BALANCE',
 'TOTAL_BL_BALANCE',
 'TOTAL_CD_BALANCE',
 'TOTAL_AL_BALANCE',
 'TOTAL_HL_BALANCE',
 'TOTAL_CC_BALANCE',
 'TOTAL_SANC_AMT',
 'TOTAL_PL_SANC_AMT',
 'TOTAL_BL_SANC_AMT',
 'TOTAL_CD_SANC_AMT',
 

In [246]:
# user_target = spark.read.parquet("user_target.parquet")

In [247]:
# user_target.columns

['user_id', 'user_ever60_9m']

In [248]:
# total_final_features = final_features.join(
#     user_target,
#     on="user_id",
#     how="left"
# )

In [249]:
features = features.withColumnRenamed("user_id", "id")

In [250]:
len(total_final_features.columns)

236

In [251]:
total_final_features.columns

['id',
 'FIRST_PRODUCT',
 'LATEST_PRODUCT',
 'MAX_BUR_VINTAGE',
 'TOTAL_TRADES',
 'TOTAL_TRADES_1_MN',
 'TOTAL_TRADES_3_MN',
 'TOTAL_TRADES_6_MN',
 'TOTAL_TRADES_12_MN',
 'TOTAL_HL_TRADES',
 'TOTAL_GL_TRADES',
 'TOTAL_PL_TRADES',
 'TOTAL_LIVE_PL_TRADES',
 'TOTAL_PL_TRADES_1_MN',
 'TOTAL_PL_TRADES_3_MN',
 'TOTAL_PL_TRADES_6_MN',
 'TOTAL_PL_TRADES_12_MN',
 'TOTAL_CC_TRADES',
 'TOTAL_CC_TRADES_1_MN',
 'TOTAL_CC_TRADES_3_MN',
 'TOTAL_CC_TRADES_6_MN',
 'TOTAL_CC_TRADES_12_MN',
 'TOTAL_SEC_TRADES',
 'TOTAL_UNSEC_TRADES',
 'TOTAL_UNSEC_TRADES_1_MN',
 'TOTAL_UNSEC_TRADES_3_MN',
 'TOTAL_UNSEC_TRADES_6_MN',
 'TOTAL_UNSEC_TRADES_12_MN',
 'TOTAL_INSTALLMENT_TRADES',
 'TOTAL_LIVE_TRADES_M0_M2',
 'TOTAL_LIVE_UNSEC_TRADES_M0_M2',
 'TOTAL_LIVE_INSTALLMENT_TRADES_M0_M2',
 'TOTAL_BALANCE',
 'TOTAL_PL_BALANCE',
 'TOTAL_BL_BALANCE',
 'TOTAL_CD_BALANCE',
 'TOTAL_AL_BALANCE',
 'TOTAL_HL_BALANCE',
 'TOTAL_CC_BALANCE',
 'TOTAL_SANC_AMT',
 'TOTAL_PL_SANC_AMT',
 'TOTAL_BL_SANC_AMT',
 'TOTAL_CD_SANC_AMT',
 'TOTA

In [253]:
# total_final_features.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in total_final_features.columns]).show()

26/06/26 23:30:07 WARN DAGScheduler: Broadcasting large task binary with size 1694.0 KiB
26/06/26 23:30:49 WARN DAGScheduler: Broadcasting large task binary with size 4.5 MiB
26/06/26 23:31:22 WARN DAGScheduler: Broadcasting large task binary with size 5.0 MiB


+---+-------------+--------------+---------------+------------+-----------------+-----------------+-----------------+------------------+---------------+---------------+---------------+--------------------+--------------------+--------------------+--------------------+---------------------+---------------+--------------------+--------------------+--------------------+---------------------+----------------+------------------+-----------------------+-----------------------+-----------------------+------------------------+------------------------+-----------------------+-----------------------------+-----------------------------------+-------------+----------------+----------------+----------------+----------------+----------------+----------------+--------------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+------------------+------------------+----------------------+----------+----------+-----+-----+-----+-----+-----+-----+---

In [257]:
# from pyspark.sql.types import NumericType

# # Get list of numeric column names
# numeric_cols = [f.name for f in total_final_features.schema.fields if isinstance(f.dataType, NumericType)]

# # Fill nulls with 0 only in those columns
# total_final_features = total_final_features.fillna(0, subset=numeric_cols)

In [258]:
# total_final_features = total_final_features.fillna(0)

In [259]:
# total_final_features.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in total_final_features.columns]).show()

26/06/26 23:39:47 WARN DAGScheduler: Broadcasting large task binary with size 1665.5 KiB
26/06/26 23:40:31 WARN DAGScheduler: Broadcasting large task binary with size 4.3 MiB
26/06/26 23:40:58 WARN DAGScheduler: Broadcasting large task binary with size 4.7 MiB


+---+-------------+--------------+---------------+------------+-----------------+-----------------+-----------------+------------------+---------------+---------------+---------------+--------------------+--------------------+--------------------+--------------------+---------------------+---------------+--------------------+--------------------+--------------------+---------------------+----------------+------------------+-----------------------+-----------------------+-----------------------+------------------------+------------------------+-----------------------+-----------------------------+-----------------------------------+-------------+----------------+----------------+----------------+----------------+----------------+----------------+--------------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+------------------+------------------+----------------------+----------+----------+-----+-----+-----+-----+-----+-----+---

In [ ]:
# final_features.coalesce(1).write.mode("overwrite").parquet(
#     "final_features.parquet"
# )

In [ ]:
# final_features = spark.read.parquet("final_features.parquet")